# Project Mythos Standalone Kaggle Pipeline

Upload this `.ipynb` by itself. It embeds the current `src/mythos` package, writes it into `/kaggle/working/project_mythos_embedded/src`, then runs the plan-aligned ARC pipeline and writes `/kaggle/working/submission.json`.

Default mode is `pipeline` + `fallback`, with internet downloads disabled. It autodiscovers pre-staged Kaggle inputs when present and otherwise uses explicit fallback adapters for missing model stages.

## 1. Bootstrap Embedded Project Mythos Code

In [ ]:
from pathlib import Path
import os
import sys

EMBEDDED_FILES = {'src/mythos/__init__.py': '"""Project Mythos ARC testing harness."""\n\nfrom mythos.arc import ArcExample, ArcTask, ArcValidationError, load_challenges\nfrom mythos.submission import Prediction, TestPrediction\n\n__all__ = [\n    "ArcExample",\n    "ArcTask",\n    "ArcValidationError",\n    "Prediction",\n    "TestPrediction",\n    "load_challenges",\n]\n', 'src/mythos/__main__.py': '"""Top-level module help for `python -m mythos`."""\n\nfrom __future__ import annotations\n\n\ndef main() -> int:\n    print(\n        "Project Mythos commands:\\n"\n        "  python -m mythos.validate data/toy/challenges.json\\n"\n        "  python -m mythos.solve --solver fixture --challenges data/toy/challenges.json --out runs/submission.json\\n"\n        "  python -m mythos.score --pred runs/submission.json --solutions data/toy/solutions.json\\n"\n        "  python -m mythos.hrm_smoke --task data/toy/challenges.json"\n    )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/arc.py': '"""ARC JSON loading and validation."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport json\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Mapping, Optional, Tuple\n\nGrid = List[List[int]]\nSolutionMap = Dict[str, Tuple[Grid, ...]]\n\nMAX_GRID_ROWS = 30\nMAX_GRID_COLS = 30\nMIN_CELL_VALUE = 0\nMAX_CELL_VALUE = 9\n\n\nclass ArcValidationError(ValueError):\n    """Raised when ARC-style data is malformed."""\n\n\n@dataclass(frozen=True)\nclass ArcExample:\n    input: Grid\n    output: Optional[Grid] = None\n\n\n@dataclass(frozen=True)\nclass ArcTask:\n    id: str\n    train: Tuple[ArcExample, ...]\n    test: Tuple[ArcExample, ...]\n\n\ndef _json_load(path: str | Path) -> Any:\n    file_path = Path(path)\n    try:\n        with file_path.open("r", encoding="utf-8") as handle:\n            return json.load(handle)\n    except json.JSONDecodeError as exc:\n        raise ArcValidationError(f"{file_path} is not valid JSON: {exc}") from exc\n\n\ndef validate_grid(value: Any, *, field: str = "grid") -> Grid:\n    """Validate and copy an ARC grid."""\n\n    if not isinstance(value, list) or not value:\n        raise ArcValidationError(f"{field} must be a non-empty list of rows")\n    if len(value) > MAX_GRID_ROWS:\n        raise ArcValidationError(f"{field} has {len(value)} rows; max is {MAX_GRID_ROWS}")\n\n    rows: Grid = []\n    expected_width: int | None = None\n    for row_idx, row in enumerate(value):\n        if not isinstance(row, list) or not row:\n            raise ArcValidationError(f"{field}[{row_idx}] must be a non-empty list")\n        if expected_width is None:\n            expected_width = len(row)\n            if expected_width > MAX_GRID_COLS:\n                raise ArcValidationError(\n                    f"{field} has {expected_width} columns; max is {MAX_GRID_COLS}"\n                )\n        elif len(row) != expected_width:\n            raise ArcValidationError(\n                f"{field} must be rectangular; row 0 has {expected_width} columns "\n                f"but row {row_idx} has {len(row)}"\n            )\n\n        copied_row: List[int] = []\n        for col_idx, cell in enumerate(row):\n            if isinstance(cell, bool) or not isinstance(cell, int):\n                raise ArcValidationError(f"{field}[{row_idx}][{col_idx}] must be an integer")\n            if cell < MIN_CELL_VALUE or cell > MAX_CELL_VALUE:\n                raise ArcValidationError(\n                    f"{field}[{row_idx}][{col_idx}]={cell}; expected 0..9"\n                )\n            copied_row.append(cell)\n        rows.append(copied_row)\n    return rows\n\n\ndef _parse_example(raw: Any, *, task_id: str, split: str, index: int, require_output: bool) -> ArcExample:\n    if not isinstance(raw, Mapping):\n        raise ArcValidationError(f"{task_id}.{split}[{index}] must be an object")\n    if "input" not in raw:\n        raise ArcValidationError(f"{task_id}.{split}[{index}] is missing input")\n    output = raw.get("output")\n    if require_output and output is None:\n        raise ArcValidationError(f"{task_id}.{split}[{index}] is missing output")\n    return ArcExample(\n        input=validate_grid(raw["input"], field=f"{task_id}.{split}[{index}].input"),\n        output=validate_grid(output, field=f"{task_id}.{split}[{index}].output")\n        if output is not None\n        else None,\n    )\n\n\ndef _parse_examples(\n    raw_examples: Any, *, task_id: str, split: str, require_output: bool\n) -> Tuple[ArcExample, ...]:\n    if not isinstance(raw_examples, list) or not raw_examples:\n        raise ArcValidationError(f"{task_id}.{split} must be a non-empty list")\n    return tuple(\n        _parse_example(\n            raw_example,\n            task_id=task_id,\n            split=split,\n            index=index,\n            require_output=require_output,\n        )\n        for index, raw_example in enumerate(raw_examples)\n    )\n\n\ndef parse_task(task_id: str, raw_task: Any) -> ArcTask:\n    if not isinstance(raw_task, Mapping):\n        raise ArcValidationError(f"{task_id} must be an object")\n    if "train" not in raw_task or "test" not in raw_task:\n        raise ArcValidationError(f"{task_id} must contain train and test splits")\n    return ArcTask(\n        id=task_id,\n        train=_parse_examples(raw_task["train"], task_id=task_id, split="train", require_output=True),\n        test=_parse_examples(raw_task["test"], task_id=task_id, split="test", require_output=False),\n    )\n\n\ndef load_challenges(path: str | Path) -> Dict[str, ArcTask]:\n    """Load an ARC challenge JSON file keyed by task id."""\n\n    raw = _json_load(path)\n    if not isinstance(raw, Mapping) or not raw:\n        raise ArcValidationError("challenge file must be a non-empty object keyed by task id")\n    return {str(task_id): parse_task(str(task_id), raw_task) for task_id, raw_task in raw.items()}\n\n\ndef grid_shape(grid: Grid) -> Tuple[int, int]:\n    return len(grid), len(grid[0])\n\n\ndef grid_equal(left: Grid, right: Grid) -> bool:\n    return left == right\n\n\ndef copy_grid(grid: Grid) -> Grid:\n    return [row[:] for row in grid]\n\n\ndef _looks_like_grid(value: Any) -> bool:\n    try:\n        validate_grid(value)\n    except ArcValidationError:\n        return False\n    return True\n\n\ndef _solution_grids_from_value(task_id: str, value: Any) -> Tuple[Grid, ...]:\n    if isinstance(value, Mapping):\n        if "test" in value:\n            return tuple(\n                validate_grid(item["output"], field=f"{task_id}.test[{idx}].output")\n                for idx, item in enumerate(value["test"])\n                if isinstance(item, Mapping) and "output" in item\n            )\n        if "output" in value:\n            return (validate_grid(value["output"], field=f"{task_id}.output"),)\n    if _looks_like_grid(value):\n        return (validate_grid(value, field=f"{task_id}.output"),)\n    if isinstance(value, list):\n        grids: List[Grid] = []\n        for idx, item in enumerate(value):\n            if isinstance(item, Mapping) and "output" in item:\n                grids.append(validate_grid(item["output"], field=f"{task_id}[{idx}].output"))\n            elif _looks_like_grid(item):\n                grids.append(validate_grid(item, field=f"{task_id}[{idx}]"))\n            else:\n                raise ArcValidationError(f"{task_id}[{idx}] is not a solution grid")\n        if grids:\n            return tuple(grids)\n    raise ArcValidationError(f"{task_id} does not contain solution outputs")\n\n\ndef load_solutions(path: str | Path) -> SolutionMap:\n    raw = _json_load(path)\n    if not isinstance(raw, Mapping) or not raw:\n        raise ArcValidationError("solution file must be a non-empty object keyed by task id")\n    return {\n        str(task_id): _solution_grids_from_value(str(task_id), value)\n        for task_id, value in raw.items()\n    }\n\n\ndef attach_solutions(tasks: Mapping[str, ArcTask], solutions: SolutionMap) -> Dict[str, ArcTask]:\n    """Return tasks with test outputs filled from a solution map."""\n\n    attached: Dict[str, ArcTask] = {}\n    for task_id, task in tasks.items():\n        if task_id not in solutions:\n            raise ArcValidationError(f"missing solutions for task {task_id}")\n        if len(solutions[task_id]) != len(task.test):\n            raise ArcValidationError(\n                f"{task_id} has {len(task.test)} test items but "\n                f"{len(solutions[task_id])} solution outputs"\n            )\n        attached[task_id] = ArcTask(\n            id=task.id,\n            train=task.train,\n            test=tuple(\n                ArcExample(input=example.input, output=solutions[task_id][index])\n                for index, example in enumerate(task.test)\n            ),\n        )\n    return attached\n\n\ndef require_test_outputs(tasks: Iterable[ArcTask]) -> None:\n    for task in tasks:\n        for index, example in enumerate(task.test):\n            if example.output is None:\n                raise ArcValidationError(f"{task.id}.test[{index}] is missing output")\n', 'src/mythos/features.py': '"""Deterministic ARC feature encoders shared by training and pipeline stages."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass\nfrom math import sqrt\nfrom typing import Iterable, Iterator, Sequence\n\nfrom mythos.arc import ArcTask, Grid\n\nARC_MAX_SIZE = 30\nARC_NUM_COLORS = 10\nDEFAULT_JEPA_FEATURE_DIM = 1280\nDEFAULT_HRM_FEATURE_DIM = 768\nDEFAULT_RULE_DIM = 4\n\n\n@dataclass(frozen=True)\nclass GridPair:\n    task_id: str\n    index: int\n    input: Grid\n    output: Grid\n\n\ndef iter_train_grid_pairs(tasks: Iterable[ArcTask]) -> Iterator[GridPair]:\n    """Yield supervised input/output pairs from ARC train examples."""\n\n    for task in tasks:\n        for index, example in enumerate(task.train):\n            if example.output is None:\n                continue\n            yield GridPair(\n                task_id=task.id,\n                index=index,\n                input=example.input,\n                output=example.output,\n            )\n\n\ndef iter_all_supervised_grid_pairs(tasks: Iterable[ArcTask]) -> Iterator[GridPair]:\n    """Yield train pairs plus test pairs whose labels are attached."""\n\n    for task in tasks:\n        yield from iter_train_grid_pairs((task,))\n        for index, example in enumerate(task.test):\n            if example.output is None:\n                continue\n            yield GridPair(\n                task_id=task.id,\n                index=index,\n                input=example.input,\n                output=example.output,\n            )\n\n\ndef grid_to_feature_vector(grid: Grid, dim: int = DEFAULT_JEPA_FEATURE_DIM) -> tuple[float, ...]:\n    """Encode a grid as a fixed-length numeric vector.\n\n    This is intentionally deterministic and dependency-free. On Kaggle it acts\n    as the local ARC grid-to-token bridge when the external I-JEPA model cannot\n    be invoked directly, and as a stable target for projection/world-model smoke\n    training.\n    """\n\n    if dim <= 0:\n        raise ValueError("feature dimension must be positive")\n\n    values = [0.0 for _ in range(dim)]\n    height = len(grid)\n    width = len(grid[0])\n    area = float(height * width)\n    flat = [cell for row in grid for cell in row]\n    counts = Counter(flat)\n\n    def add(index: int, value: float) -> None:\n        if 0 <= index < dim:\n            values[index] += float(value)\n\n    add(0, height / ARC_MAX_SIZE)\n    add(1, width / ARC_MAX_SIZE)\n    add(2, area / float(ARC_MAX_SIZE * ARC_MAX_SIZE))\n    add(3, sum(1 for cell in flat if cell != 0) / area)\n    add(4, sum(flat) / (9.0 * area))\n\n    for color in range(ARC_NUM_COLORS):\n        add(8 + color, counts.get(color, 0) / area)\n\n    row_offset = 32\n    for row_index in range(ARC_MAX_SIZE):\n        if row_index < height:\n            row = grid[row_index]\n            row_area = float(width)\n            add(row_offset + row_index, sum(row) / (9.0 * row_area))\n            add(row_offset + ARC_MAX_SIZE + row_index, sum(1 for cell in row if cell != 0) / row_area)\n\n    col_offset = row_offset + 2 * ARC_MAX_SIZE\n    for col_index in range(ARC_MAX_SIZE):\n        if col_index < width:\n            column = [grid[row_index][col_index] for row_index in range(height)]\n            col_area = float(height)\n            add(col_offset + col_index, sum(column) / (9.0 * col_area))\n            add(col_offset + ARC_MAX_SIZE + col_index, sum(1 for cell in column if cell != 0) / col_area)\n\n    flat_offset = col_offset + 2 * ARC_MAX_SIZE\n    hash_offset = max(flat_offset, int(dim * 0.875))\n    flat_capacity = max(0, min(hash_offset, dim) - flat_offset)\n    for row_index in range(ARC_MAX_SIZE):\n        for col_index in range(ARC_MAX_SIZE):\n            flat_index = row_index * ARC_MAX_SIZE + col_index\n            if flat_index >= flat_capacity:\n                break\n            target_index = flat_offset + flat_index\n            if row_index < height and col_index < width:\n                add(target_index, (grid[row_index][col_index] + 1) / 10.0)\n            else:\n                add(target_index, 0.0)\n\n    hash_dim = dim - hash_offset\n    if hash_dim > 0:\n        for row_index, row in enumerate(grid):\n            for col_index, color in enumerate(row):\n                base = row_index * 1009 + col_index * 917 + color * 613\n                magnitude = ((color + 1) / 10.0) * (1.0 + (row_index + col_index) / 60.0)\n                for salt in range(4):\n                    slot = hash_offset + ((base + salt * 193) % hash_dim)\n                    sign = 1.0 if ((base + salt * 389) % 2 == 0) else -1.0\n                    add(slot, sign * magnitude)\n\n    return _l2_normalize(values)\n\n\ndef task_rule_vector(task: ArcTask, dim: int = DEFAULT_RULE_DIM) -> tuple[float, ...]:\n    """Summarize the demonstrated transformation as a fixed-length rule vector."""\n\n    if dim <= 0:\n        raise ValueError("rule dimension must be positive")\n\n    shape_deltas: list[tuple[int, int]] = []\n    density_deltas: list[float] = []\n    color_overlaps: list[float] = []\n    color_shift_votes: list[float] = []\n\n    for example in task.train:\n        if example.output is None:\n            continue\n        in_h, in_w = len(example.input), len(example.input[0])\n        out_h, out_w = len(example.output), len(example.output[0])\n        shape_deltas.append((out_h - in_h, out_w - in_w))\n        density_deltas.append(_density(example.output) - _density(example.input))\n        color_overlaps.append(_color_jaccard(example.input, example.output))\n        color_shift_votes.append(_dominant_color(example.output) - _dominant_color(example.input))\n\n    base = [\n        _average(delta[0] for delta in shape_deltas) / ARC_MAX_SIZE,\n        _average(delta[1] for delta in shape_deltas) / ARC_MAX_SIZE,\n        _average(density_deltas),\n        _average(color_overlaps),\n        _average(color_shift_votes) / 9.0,\n    ]\n    if dim <= len(base):\n        return tuple(round(value, 6) for value in base[:dim])\n\n    values = [0.0 for _ in range(dim)]\n    for index, value in enumerate(base):\n        values[index] = value\n    for pair_index, example in enumerate(task.train):\n        if example.output is None:\n            continue\n        for grid_index, grid in enumerate((example.input, example.output)):\n            for color, count in Counter(cell for row in grid for cell in row).items():\n                slot = len(base) + ((pair_index * 97 + grid_index * 31 + color * 17) % (dim - len(base)))\n                values[slot] += count / 900.0\n    return tuple(round(value, 6) for value in values)\n\n\ndef grid_to_hrm_sequence(grid: Grid, *, max_size: int = ARC_MAX_SIZE) -> tuple[int, ...]:\n    """Encode a grid using HRM ARC token conventions: PAD=0, EOS=1, colors=2..11."""\n\n    height = len(grid)\n    width = len(grid[0])\n    if height > max_size or width > max_size:\n        raise ValueError(f"grid shape {height}x{width} exceeds {max_size}x{max_size}")\n    tokens = [[0 for _ in range(max_size)] for _ in range(max_size)]\n    for row_index, row in enumerate(grid):\n        for col_index, color in enumerate(row):\n            tokens[row_index][col_index] = color + 2\n    if height < max_size:\n        for col_index in range(width):\n            tokens[height][col_index] = 1\n    if width < max_size:\n        for row_index in range(height):\n            tokens[row_index][width] = 1\n    return tuple(token for row in tokens for token in row)\n\n\ndef hrm_sequence_to_grid(\n    sequence: Sequence[int],\n    *,\n    shape_hint: tuple[int, int] | None = None,\n    max_size: int = ARC_MAX_SIZE,\n) -> Grid:\n    """Decode a HRM ARC token sequence into a valid 0..9 grid."""\n\n    if len(sequence) != max_size * max_size:\n        raise ValueError(f"expected {max_size * max_size} HRM tokens, got {len(sequence)}")\n    matrix = [\n        [int(sequence[row * max_size + col]) for col in range(max_size)]\n        for row in range(max_size)\n    ]\n    if shape_hint is None:\n        shape_hint = _infer_shape_from_eos(matrix, max_size=max_size)\n    height = min(max(1, shape_hint[0]), max_size)\n    width = min(max(1, shape_hint[1]), max_size)\n    return [\n        [_token_to_color(matrix[row][col]) for col in range(width)]\n        for row in range(height)\n    ]\n\n\ndef output_shape_hint(task: ArcTask, input_grid: Grid) -> tuple[int, int] | None:\n    """Choose an output shape only when train examples agree on one.\n\n    When train output shapes disagree (common for crop/symmetry-repair\n    tasks, where the output size is the bounding box of some occluded\n    region and varies per example), there is no safe static guess -- return\n    None so the caller falls back to the model\'s own predicted EOS boundary\n    markers (see `_infer_shape_from_eos`) instead of forcing the full input\n    grid\'s shape, which is virtually never correct for this task family and\n    makes exact-match scoring impossible regardless of prediction quality.\n    """\n\n    shapes = {\n        (len(example.output), len(example.output[0]))\n        for example in task.train\n        if example.output is not None\n    }\n    if len(shapes) == 1:\n        return next(iter(shapes))\n    return None\n\n\ndef embedding_cosine_similarity(left: Sequence[float], right: Sequence[float]) -> float:\n    numerator = sum(a * b for a, b in zip(left, right))\n    left_norm = sqrt(sum(a * a for a in left))\n    right_norm = sqrt(sum(b * b for b in right))\n    if left_norm == 0.0 or right_norm == 0.0:\n        return 0.0\n    return numerator / (left_norm * right_norm)\n\n\ndef max_pairwise_cosine(vectors: Sequence[Sequence[float]]) -> float:\n    if len(vectors) < 2:\n        return 0.0\n    max_value = -1.0\n    for left_index, left in enumerate(vectors):\n        for right in vectors[left_index + 1 :]:\n            max_value = max(max_value, embedding_cosine_similarity(left, right))\n    return max_value\n\n\ndef _infer_shape_from_eos(matrix: Sequence[Sequence[int]], *, max_size: int) -> tuple[int, int]:\n    row_scores = [\n        sum(1 for value in matrix[row_index] if value == 1)\n        for row_index in range(max_size)\n    ]\n    col_scores = [\n        sum(1 for row_index in range(max_size) if matrix[row_index][col_index] == 1)\n        for col_index in range(max_size)\n    ]\n    height = _best_boundary_index(row_scores, max_size)\n    width = _best_boundary_index(col_scores, max_size)\n    return max(1, height), max(1, width)\n\n\ndef _best_boundary_index(scores: Sequence[int], default: int, *, min_score: int = 2) -> int:\n    """Pick the index most likely to be the EOS boundary marker.\n\n    Per `grid_to_hrm_sequence`, every in-grid row/column already carries\n    exactly one legitimate stray EOS(=1) token from the *orthogonal*\n    boundary marker (the EOS column marks every content row; the EOS row\n    marks every content column). A low fixed threshold is therefore\n    trivially false-triggered by that baseline plus a single unit of\n    prediction noise on some earlier row/column. The genuine boundary\n    instead carries a full run of EOS tokens -- one per in-grid cell along\n    that axis -- so picking the argmax (not the first index crossing a low\n    threshold) is far more robust to noisy raw predictions.\n    """\n\n    best_index = default\n    best_score = min_score - 1\n    for index, score in enumerate(scores):\n        if score > best_score:\n            best_score = score\n            best_index = index\n    return best_index if best_score >= min_score else default\n\n\ndef _token_to_color(token: int) -> int:\n    if token <= 1:\n        return 0\n    return min(max(token - 2, 0), 9)\n\n\ndef _density(grid: Grid) -> float:\n    flat = [cell for row in grid for cell in row]\n    return sum(1 for cell in flat if cell != 0) / float(len(flat))\n\n\ndef _dominant_color(grid: Grid) -> int:\n    counts = Counter(cell for row in grid for cell in row)\n    return counts.most_common(1)[0][0]\n\n\ndef _color_jaccard(left: Grid, right: Grid) -> float:\n    left_colors = {cell for row in left for cell in row}\n    right_colors = {cell for row in right for cell in row}\n    union = left_colors | right_colors\n    if not union:\n        return 1.0\n    return len(left_colors & right_colors) / len(union)\n\n\ndef _average(values: Iterable[float]) -> float:\n    collected = list(values)\n    return sum(collected) / len(collected) if collected else 0.0\n\n\ndef _l2_normalize(values: Sequence[float]) -> tuple[float, ...]:\n    norm = sqrt(sum(value * value for value in values))\n    if norm == 0.0:\n        return tuple(round(value, 6) for value in values)\n    return tuple(round(value / norm, 6) for value in values)\n', 'src/mythos/hrm_dataset.py': '"""Dataset-preparation glue for the external HRM checkout."""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nfrom typing import Iterable\n\nfrom mythos.arc import ArcTask, ArcValidationError, Grid, require_test_outputs\n\n\ndef default_run_dir() -> Path:\n    import os\n\n    # Must be absolute: build_hrm_dataset() invokes HRM\'s dataset builder with\n    # cwd=repo_dir (the external HRM checkout), while the caller that later reads\n    # the built dataset back (HRMInferenceRunner._run_external_evaluate, running in\n    # the notebook/CLI\'s own process) has a different cwd. A relative path here\n    # resolves to two different real locations across those processes -- confirmed\n    # by a real run: the builder wrote train/dataset.json under repo_dir, but the\n    # dataloader looked for it relative to the notebook\'s cwd and got FileNotFoundError.\n    return Path(os.environ.get("MYTHOS_RUN_DIR", "runs")).resolve()\n\n\ndef prepare_hrm_raw_dataset(\n    tasks: Iterable[ArcTask],\n    output_dir: str | Path,\n    *,\n    allow_dummy_test_outputs: bool = False,\n) -> Path:\n    """Write tasks into the directory shape HRM\'s ARC dataset builder expects."""\n\n    task_list = list(tasks)\n    if not allow_dummy_test_outputs:\n        require_test_outputs(task_list)\n\n    raw_data_dir = Path(output_dir)\n    eval_dir = raw_data_dir / "evaluation"\n    eval_dir.mkdir(parents=True, exist_ok=True)\n\n    for task in task_list:\n        raw_task = {\n            "train": [\n                {"input": example.input, "output": example.output}\n                for example in task.train\n            ],\n            "test": [\n                {\n                    "input": example.input,\n                    "output": example.output\n                    if example.output is not None\n                    else _dummy_output_like(example.input),\n                }\n                for example in task.test\n            ],\n        }\n        with (eval_dir / f"{task.id}.json").open("w", encoding="utf-8") as handle:\n            json.dump(raw_task, handle, indent=2)\n            handle.write("\\n")\n    return raw_data_dir\n\n\ndef _dummy_output_like(grid: Grid) -> Grid:\n    return [[0 for _ in row] for row in grid]\n\n\ndef build_hrm_dataset(\n    *,\n    hrm_repo_dir: str | Path,\n    raw_data_dir: str | Path,\n    output_dir: str | Path,\n    num_aug: int = 0,\n) -> subprocess.CompletedProcess[str]:\n    """Invoke HRM\'s own ARC dataset builder against a prepared raw-data directory."""\n\n    repo_dir = Path(hrm_repo_dir)\n    script = repo_dir / "dataset" / "build_arc_dataset.py"\n    if not script.exists():\n        raise ArcValidationError(f"HRM dataset builder not found: {script}")\n\n    output_path = Path(output_dir)\n    output_path.mkdir(parents=True, exist_ok=True)\n\n    # build_arc_dataset.py\'s DataProcessConfig.dataset_dirs is a List[str] Pydantic\n    # field defaulting to ["dataset/raw-data/ARC-AGI/data", "dataset/raw-data/ConceptARC/corpus"],\n    # resolved relative to the subprocess\'s cwd. Passing --dataset-dirs <path> on the CLI\n    # does not override this list -- verified against a real run: it silently kept\n    # scanning the unmodified default and crashed with FileNotFoundError. Sidestep the CLI\n    # entirely by giving the subprocess a writable cwd that already has the default paths\n    # satisfied, decoupled from repo_dir -- which is read-only when HRM_REPO_DIR points at\n    # a Kaggle Dataset mount, confirmed by a real run failing with OSError(30, \'Read-only\n    # file system\') when this used to write directly under repo_dir. The script itself is\n    # still invoked from its real (possibly read-only) location via an absolute path.\n    build_cwd = output_path.parent / "hrm_build_cwd"\n    default_arc_dir = build_cwd / "dataset" / "raw-data" / "ARC-AGI" / "data"\n    default_concept_dir = build_cwd / "dataset" / "raw-data" / "ConceptARC" / "corpus"\n    if default_arc_dir.is_symlink() or default_arc_dir.is_file():\n        default_arc_dir.unlink()\n    elif default_arc_dir.exists():\n        shutil.rmtree(default_arc_dir)\n    default_arc_dir.parent.mkdir(parents=True, exist_ok=True)\n    shutil.copytree(Path(raw_data_dir), default_arc_dir)\n    default_concept_dir.mkdir(parents=True, exist_ok=True)\n\n    command = [\n        sys.executable,\n        str(script.resolve()),\n        "--output-dir",\n        str(output_path),\n        "--num-aug",\n        str(num_aug),\n    ]\n    return subprocess.run(\n        command,\n        cwd=build_cwd,\n        check=True,\n        capture_output=True,\n        text=True,\n    )\n', 'src/mythos/hrm_smoke.py': '"""CLI smoke test for the external HRM runtime."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom mythos.arc import ArcValidationError, attach_solutions, load_challenges, load_solutions\nfrom mythos.hrm_dataset import build_hrm_dataset, default_run_dir, prepare_hrm_raw_dataset\nfrom mythos.solvers.hrm import HRMEnvironment, HRMEnvironmentError\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Smoke-test external HRM integration.")\n    parser.add_argument("--task", required=True, help="ARC-style challenge JSON for the smoke run.")\n    parser.add_argument("--solutions", help="Optional solution JSON if --task omits test outputs.")\n    parser.add_argument("--run-dir", default=None, help="Output directory for smoke artifacts.")\n    parser.add_argument("--num-aug", type=int, default=0, help="HRM dataset-builder augmentation count.")\n    parser.add_argument(\n        "--skip-dataset-build",\n        action="store_true",\n        help="Only write the HRM raw-data layout; do not invoke HRM\'s dataset builder.",\n    )\n    args = parser.parse_args(argv)\n\n    started = time.perf_counter()\n    try:\n        tasks = load_challenges(args.task)\n        if args.solutions:\n            tasks = attach_solutions(tasks, load_solutions(args.solutions))\n\n        env = HRMEnvironment.from_env()\n        env.validate(require_cuda=True)\n        modules = env.import_modules()\n\n        torch = HRMEnvironment._import_torch()\n        torch.cuda.reset_peak_memory_stats()\n        checkpoint = env.load_checkpoint()\n\n        run_dir = Path(args.run_dir) if args.run_dir else default_run_dir() / "hrm_smoke"\n        raw_dir = prepare_hrm_raw_dataset(tasks.values(), run_dir / "raw" / "ARC-AGI-2" / "data")\n\n        dataset_build = None\n        if not args.skip_dataset_build:\n            result = build_hrm_dataset(\n                hrm_repo_dir=env.repo_dir,\n                raw_data_dir=raw_dir,\n                output_dir=run_dir / "data" / "arc-2-smoke",\n                num_aug=args.num_aug,\n            )\n            dataset_build = {\n                "returncode": result.returncode,\n                "stdout_tail": result.stdout[-2000:],\n                "stderr_tail": result.stderr[-2000:],\n            }\n    except (ArcValidationError, HRMEnvironmentError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n\n    summary = {\n        "tasks": len(tasks),\n        "hrm_repo_dir": str(env.repo_dir),\n        "checkpoint_path": str(env.checkpoint_path),\n        "checkpoint_type": type(checkpoint).__name__,\n        "imported_modules": sorted(modules),\n        "raw_data_dir": str(raw_dir),\n        "dataset_build": dataset_build,\n        "elapsed_seconds": round(time.perf_counter() - started, 3),\n        "cuda_peak_memory_bytes": int(torch.cuda.max_memory_allocated()),\n    }\n    print(json.dumps(summary, indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/jepa_encoder.py': '"""Optional transformers-native I-JEPA image encoder for ARC grids."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nfrom mythos.arc import Grid\n\nARC_PALETTE: dict[int, tuple[int, int, int]] = {\n    0: (0, 0, 0),\n    1: (0, 116, 217),\n    2: (255, 65, 54),\n    3: (46, 204, 64),\n    4: (255, 220, 0),\n    5: (170, 170, 170),\n    6: (240, 18, 190),\n    7: (255, 133, 27),\n    8: (127, 219, 255),\n    9: (135, 12, 37),\n}\n\n\nclass JepaEncodingError(RuntimeError):\n    """Raised when optional I-JEPA image encoding cannot run."""\n\n\ndef grid_to_rgb_image(grid: Grid, *, cell_px: int = 8, output_size: int = 224):\n    """Rasterize an ARC grid to the RGB image shape expected by ViT-H/14."""\n\n    try:\n        from PIL import Image\n    except Exception as exc:  # pragma: no cover - optional dependency.\n        raise JepaEncodingError("Pillow is required for real I-JEPA grid rasterization") from exc\n\n    if cell_px <= 0:\n        raise ValueError("cell_px must be positive")\n    height = len(grid)\n    width = len(grid[0])\n    image = Image.new("RGB", (width * cell_px, height * cell_px))\n    for row_index, row in enumerate(grid):\n        for col_index, color in enumerate(row):\n            image.paste(\n                ARC_PALETTE[int(color)],\n                (\n                    col_index * cell_px,\n                    row_index * cell_px,\n                    (col_index + 1) * cell_px,\n                    (row_index + 1) * cell_px,\n                ),\n            )\n    resampling = getattr(Image, "Resampling", Image)\n    return image.resize((output_size, output_size), resampling.NEAREST)\n\n\n@dataclass\nclass JepaImageEncoder:\n    """Frozen transformers I-JEPA feature extractor."""\n\n    model_root: Path\n    device: str\n    processor: object\n    model: object\n\n    @classmethod\n    def from_path(cls, model_path: str | Path, *, device: str | None = None) -> "JepaImageEncoder":\n        try:\n            import torch\n            from transformers import AutoModel, AutoProcessor\n        except Exception as exc:  # pragma: no cover - optional dependency.\n            raise JepaEncodingError("transformers, torch, and Pillow are required for real I-JEPA") from exc\n\n        root = _model_root(model_path)\n        selected_device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n        try:\n            processor = AutoProcessor.from_pretrained(root, local_files_only=True)\n            model = AutoModel.from_pretrained(root, local_files_only=True).to(selected_device)\n            model.eval()\n            for parameter in model.parameters():\n                parameter.requires_grad = False\n        except Exception as exc:  # pragma: no cover - depends on external model files.\n            raise JepaEncodingError(f"failed to load transformers I-JEPA model from {root}: {exc}") from exc\n        return cls(model_root=root, device=selected_device, processor=processor, model=model)\n\n    def encode_grid(self, grid: Grid) -> tuple[float, ...]:\n        try:\n            import torch\n        except Exception as exc:  # pragma: no cover - optional dependency.\n            raise JepaEncodingError("torch is required for real I-JEPA") from exc\n\n        image = grid_to_rgb_image(grid)\n        encoded = self.processor(images=image, return_tensors="pt")\n        encoded = {\n            key: value.to(self.device) if hasattr(value, "to") else value\n            for key, value in encoded.items()\n        }\n        with torch.no_grad():\n            outputs = self.model(**encoded)\n        hidden = getattr(outputs, "last_hidden_state", None)\n        if hidden is None:\n            raise JepaEncodingError("I-JEPA model output did not include last_hidden_state")\n        pooled = hidden.mean(dim=1)[0].detach().float().cpu().tolist()\n        return tuple(round(float(value), 6) for value in pooled)\n\n\ndef _model_root(model_path: str | Path) -> Path:\n    path = Path(model_path)\n    return path.parent if path.is_file() else path\n', 'src/mythos/kaggle_models.py': '"""Kaggle model input auto-discovery.\n\nKaggle submissions usually mount model code and checkpoints under\n`/kaggle/input/<dataset-name>/...`. This module scans those inputs and sets the\nenvironment variables consumed by `mythos.models.ModelRegistry`.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nimport os\nimport subprocess\nimport urllib.request\nfrom typing import Iterable\n\n\nCHECKPOINT_SUFFIXES = (".pt", ".pth", ".ckpt", ".bin", ".safetensors", ".tar")\n\n\nMODEL_ENV_KEYS = (\n    "IJEPA_CHECKPOINT_PATH",\n    "IJEPA_PROJECTION_CHECKPOINT_PATH",\n    "HRM_TEXT_REPO_DIR",\n    "HRM_TEXT_CHECKPOINT_PATH",\n    "WORLD_MODEL_CHECKPOINT_PATH",\n    "TTT_LORA_CHECKPOINT_PATH",\n    "HRM_REPO_DIR",\n    "HRM_CHECKPOINT_PATH",\n)\n\nHF_MODEL_SPECS = (\n    {\n        "name": "jepa",\n        "repo_id_env": "IJEPA_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "IJEPA_CHECKPOINT_PATH",\n    },\n    {\n        "name": "jepa_projection",\n        "repo_id_env": "IJEPA_PROJECTION_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "IJEPA_PROJECTION_CHECKPOINT_PATH",\n    },\n    {\n        "name": "hrm_text",\n        "repo_id_env": "HRM_TEXT_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "HRM_TEXT_CHECKPOINT_PATH",\n    },\n    {\n        "name": "world_model",\n        "repo_id_env": "WORLD_MODEL_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "WORLD_MODEL_CHECKPOINT_PATH",\n    },\n    {\n        "name": "ttt_lora",\n        "repo_id_env": "TTT_LORA_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "TTT_LORA_CHECKPOINT_PATH",\n    },\n    {\n        "name": "hrm_l_module",\n        "repo_id_env": "HRM_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "HRM_CHECKPOINT_PATH",\n    },\n)\n\nGIT_REPO_SPECS = (\n    {\n        "name": "hrm_text",\n        "url_env": "HRM_TEXT_GIT_REPO_URL",\n        "repo_dir_env": "HRM_TEXT_REPO_DIR",\n    },\n    {\n        "name": "hrm_l_module",\n        "url_env": "HRM_GIT_REPO_URL",\n        "repo_dir_env": "HRM_REPO_DIR",\n    },\n)\n\nDIRECT_CHECKPOINT_SPECS = (\n    {\n        "name": "jepa",\n        "url_env": "IJEPA_CHECKPOINT_URL",\n        "checkpoint_env": "IJEPA_CHECKPOINT_PATH",\n    },\n    {\n        "name": "jepa_projection",\n        "url_env": "IJEPA_PROJECTION_CHECKPOINT_URL",\n        "checkpoint_env": "IJEPA_PROJECTION_CHECKPOINT_PATH",\n    },\n    {\n        "name": "hrm_text",\n        "url_env": "HRM_TEXT_CHECKPOINT_URL",\n        "checkpoint_env": "HRM_TEXT_CHECKPOINT_PATH",\n    },\n    {\n        "name": "world_model",\n        "url_env": "WORLD_MODEL_CHECKPOINT_URL",\n        "checkpoint_env": "WORLD_MODEL_CHECKPOINT_PATH",\n    },\n    {\n        "name": "ttt_lora",\n        "url_env": "TTT_LORA_CHECKPOINT_URL",\n        "checkpoint_env": "TTT_LORA_CHECKPOINT_PATH",\n    },\n    {\n        "name": "hrm_l_module",\n        "url_env": "HRM_CHECKPOINT_URL",\n        "checkpoint_env": "HRM_CHECKPOINT_PATH",\n    },\n)\n\n\ndef download_git_code_repositories(\n    output_root: str | Path = "/kaggle/working/model_code",\n    *,\n    apply: bool = True,\n) -> dict[str, object]:\n    """Clone configured Git repos for model code and export repo env vars."""\n\n    result: dict[str, object] = {\n        "output_root": str(output_root),\n        "cloned": {},\n        "skipped": [],\n        "errors": {},\n    }\n    configured = [spec for spec in GIT_REPO_SPECS if os.environ.get(str(spec["url_env"]))]\n    if not configured:\n        result["skipped"] = [str(spec["url_env"]) for spec in GIT_REPO_SPECS]\n        return result\n\n    root = Path(output_root)\n    root.mkdir(parents=True, exist_ok=True)\n    for spec in configured:\n        name = str(spec["name"])\n        url = os.environ[str(spec["url_env"])]\n        repo_dir = root / name\n        try:\n            if not repo_dir.exists():\n                subprocess.run(\n                    ["git", "clone", "--depth", "1", url, str(repo_dir)],\n                    check=True,\n                    capture_output=True,\n                    text=True,\n                )\n            if apply:\n                os.environ.setdefault(str(spec["repo_dir_env"]), str(repo_dir))\n            result["cloned"][name] = {\n                "url": url,\n                "repo_dir": str(repo_dir),\n                "repo_dir_env": spec["repo_dir_env"],\n            }\n        except Exception as exc:\n            result["errors"][name] = str(exc)\n    return result\n\n\ndef download_direct_checkpoint_inputs(\n    output_root: str | Path = "/kaggle/working/model_inputs/direct",\n    *,\n    apply: bool = True,\n) -> dict[str, object]:\n    """Download configured direct checkpoint URLs and export checkpoint env vars."""\n\n    result: dict[str, object] = {\n        "output_root": str(output_root),\n        "downloaded": {},\n        "skipped": [],\n        "errors": {},\n    }\n    configured = [spec for spec in DIRECT_CHECKPOINT_SPECS if os.environ.get(str(spec["url_env"]))]\n    if not configured:\n        result["skipped"] = [str(spec["url_env"]) for spec in DIRECT_CHECKPOINT_SPECS]\n        return result\n\n    root = Path(output_root)\n    root.mkdir(parents=True, exist_ok=True)\n    for spec in configured:\n        name = str(spec["name"])\n        url = os.environ[str(spec["url_env"])]\n        target = root / name / _filename_from_url(url)\n        try:\n            target.parent.mkdir(parents=True, exist_ok=True)\n            if not target.exists():\n                urllib.request.urlretrieve(url, target)\n            checkpoint_env = str(spec["checkpoint_env"])\n            if apply:\n                os.environ.setdefault(checkpoint_env, str(target))\n            result["downloaded"][name] = {\n                "url": url,\n                "checkpoint": str(target),\n                "checkpoint_env": checkpoint_env,\n            }\n        except Exception as exc:\n            result["errors"][name] = str(exc)\n    return result\n\n\ndef download_huggingface_model_inputs(\n    output_root: str | Path = "/kaggle/working/model_inputs",\n    *,\n    apply: bool = True,\n) -> dict[str, object]:\n    """Download configured Hugging Face model repos before model loading.\n\n    Configure with env vars such as `HRM_HF_REPO_ID`. Optional env vars named\n    `<PREFIX>_HF_CHECKPOINT_GLOB` can narrow checkpoint selection, for example\n    `HRM_HF_CHECKPOINT_GLOB="*.pt"`.\n    """\n\n    result: dict[str, object] = {\n        "output_root": str(output_root),\n        "downloaded": {},\n        "skipped": [],\n        "errors": {},\n    }\n\n    configured = [spec for spec in HF_MODEL_SPECS if os.environ.get(str(spec["repo_id_env"]))]\n    if not configured:\n        result["skipped"] = [str(spec["repo_id_env"]) for spec in HF_MODEL_SPECS]\n        return result\n\n    try:\n        from huggingface_hub import snapshot_download\n    except Exception as exc:\n        result["errors"]["huggingface_hub"] = (\n            "huggingface_hub is not installed or cannot be imported: " + str(exc)\n        )\n        return result\n\n    output_dir = Path(output_root)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    for spec in configured:\n        name = str(spec["name"])\n        repo_id_env = str(spec["repo_id_env"])\n        repo_id = os.environ[repo_id_env]\n        local_dir = output_dir / name\n        glob_env = f"{repo_id_env.removesuffix(\'_REPO_ID\')}_CHECKPOINT_GLOB"\n        checkpoint_glob = os.environ.get(glob_env)\n\n        try:\n            downloaded = Path(\n                snapshot_download(\n                    repo_id=repo_id,\n                    local_dir=local_dir,\n                    local_dir_use_symlinks=False,\n                )\n            )\n            checkpoint = _find_checkpoint_by_glob(downloaded, checkpoint_glob)\n            if checkpoint is None:\n                result["errors"][name] = (\n                    f"downloaded {repo_id} to {downloaded}, but found no checkpoint "\n                    f"matching {checkpoint_glob or CHECKPOINT_SUFFIXES}"\n                )\n                continue\n\n            repo_dir_env = spec["repo_dir_env"]\n            checkpoint_env = str(spec["checkpoint_env"])\n            if apply:\n                if repo_dir_env is not None:\n                    os.environ.setdefault(str(repo_dir_env), str(downloaded))\n                os.environ.setdefault(checkpoint_env, str(checkpoint))\n\n            result["downloaded"][name] = {\n                "repo_id": repo_id,\n                "local_dir": str(downloaded),\n                "checkpoint": str(checkpoint),\n                "repo_dir_env": repo_dir_env,\n                "checkpoint_env": checkpoint_env,\n            }\n        except Exception as exc:\n            result["errors"][name] = str(exc)\n\n    return result\n\n\ndef autodiscover_model_inputs(\n    input_root: str | Path = "/kaggle/input",\n    *,\n    apply: bool = True,\n) -> dict[str, object]:\n    """Find likely model repos/checkpoints and optionally export env vars."""\n\n    root = Path(input_root)\n    result: dict[str, object] = {\n        "input_root": str(root),\n        "exists": root.exists(),\n        "set": {},\n        "missing": [],\n    }\n    if not root.exists():\n        result["missing"] = list(MODEL_ENV_KEYS)\n        return result\n\n    discovered: dict[str, Path] = {}\n\n    hrm_repo = _find_hrm_repo(root)\n    hrm_checkpoint = _find_checkpoint(root, include=("hrm",), exclude=("text", "lora", "adapter"))\n    if hrm_repo is not None and hrm_checkpoint is not None:\n        discovered["HRM_REPO_DIR"] = hrm_repo\n        discovered["HRM_CHECKPOINT_PATH"] = hrm_checkpoint\n\n    ijepa_checkpoint = _find_checkpoint(root, include=("ijepa", "i-jepa", "jepa"), exclude=("projection",))\n    if ijepa_checkpoint is not None:\n        discovered["IJEPA_CHECKPOINT_PATH"] = ijepa_checkpoint\n\n    ijepa_projection_checkpoint = _find_checkpoint(root, include=("ijepa", "projection"), require_all=True)\n    if ijepa_projection_checkpoint is not None:\n        discovered["IJEPA_PROJECTION_CHECKPOINT_PATH"] = ijepa_projection_checkpoint\n\n    hrm_text_repo = _find_named_repo(root, names=("hrm-text", "hrm_text", "hrmtext"))\n    hrm_text_checkpoint = _find_checkpoint(root, include=("hrm", "text"), require_all=True)\n    if hrm_text_repo is not None and hrm_text_checkpoint is not None:\n        discovered["HRM_TEXT_REPO_DIR"] = hrm_text_repo\n        discovered["HRM_TEXT_CHECKPOINT_PATH"] = hrm_text_checkpoint\n\n    world_model_checkpoint = _find_checkpoint(root, include=("world", "transition"))\n    if world_model_checkpoint is not None:\n        discovered["WORLD_MODEL_CHECKPOINT_PATH"] = world_model_checkpoint\n\n    lora_checkpoint = _find_checkpoint(root, include=("lora", "adapter"))\n    if lora_checkpoint is not None:\n        discovered["TTT_LORA_CHECKPOINT_PATH"] = lora_checkpoint\n\n    set_values: dict[str, str] = {}\n    for key, path in discovered.items():\n        if key in os.environ:\n            set_values[key] = os.environ[key]\n            continue\n        if apply:\n            os.environ[key] = str(path)\n        set_values[key] = str(path)\n\n    result["set"] = set_values\n    result["missing"] = [key for key in MODEL_ENV_KEYS if key not in set_values and key not in os.environ]\n    return result\n\n\ndef _find_hrm_repo(root: Path) -> Path | None:\n    for path in _iter_dirs(root):\n        if (path / "evaluate.py").exists() and (path / "dataset" / "build_arc_dataset.py").exists():\n            return path\n    return None\n\n\ndef _find_named_repo(root: Path, *, names: Iterable[str]) -> Path | None:\n    lowered_names = tuple(name.lower() for name in names)\n    for path in _iter_dirs(root):\n        path_text = path.as_posix().lower()\n        if any(name in path_text for name in lowered_names):\n            return path\n    return None\n\n\ndef _find_checkpoint(\n    root: Path,\n    *,\n    include: Iterable[str],\n    exclude: Iterable[str] = (),\n    require_all: bool = False,\n) -> Path | None:\n    include_terms = tuple(term.lower() for term in include)\n    exclude_terms = tuple(term.lower() for term in exclude)\n    candidates = []\n    for path in root.rglob("*"):\n        if not path.is_file() or path.suffix.lower() not in CHECKPOINT_SUFFIXES:\n            continue\n        path_text = path.as_posix().lower()\n        if require_all and not all(term in path_text for term in include_terms):\n            continue\n        if not require_all and not any(term in path_text for term in include_terms):\n            continue\n        if any(term in path_text for term in exclude_terms):\n            continue\n        candidates.append(path)\n    if not candidates:\n        return None\n    return sorted(candidates, key=lambda item: (len(item.as_posix()), item.as_posix()))[0]\n\n\ndef _find_checkpoint_by_glob(root: Path, checkpoint_glob: str | None) -> Path | None:\n    if checkpoint_glob:\n        candidates = [path for path in root.rglob(checkpoint_glob) if path.is_file()]\n    else:\n        candidates = [\n            path\n            for path in root.rglob("*")\n            if path.is_file() and path.suffix.lower() in CHECKPOINT_SUFFIXES\n        ]\n    if not candidates:\n        return None\n    return sorted(candidates, key=lambda item: (len(item.as_posix()), item.as_posix()))[0]\n\n\ndef _filename_from_url(url: str) -> str:\n    filename = url.rstrip("/").split("/")[-1]\n    return filename or "checkpoint.pt"\n\n\ndef _iter_dirs(root: Path):\n    for path in root.rglob("*"):\n        if path.is_dir():\n            yield path\n', 'src/mythos/kaggle_run.py': '"""Kaggle-oriented runner for producing /kaggle/working/submission.json."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport traceback\n\nfrom mythos.arc import ArcTask, ArcValidationError, copy_grid, load_challenges\nfrom mythos.metrics import score_files\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.solvers.baseline import BaselineSolver\nfrom mythos.solvers.factory import make_solver\nfrom mythos.solvers.hrm import HRMEnvironment, HRMInferenceRunner, HRMTTTRunner, TTTConfig\nfrom mythos.submission import Prediction, write_submission\n\nDEFAULT_KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")\nDEFAULT_KAGGLE_OUTPUT = Path("/kaggle/working/submission.json")\n\nCHALLENGE_FILES = {\n    "training": ("arc-agi_training-challenges.json", "arc-agi_training_challenges.json"),\n    "evaluation": ("arc-agi_evaluation-challenges.json", "arc-agi_evaluation_challenges.json"),\n    "test": ("arc-agi_test-challenges.json", "arc-agi_test_challenges.json"),\n}\n\nSOLUTION_FILES = {\n    "training": ("arc-agi_training-solutions.json", "arc-agi_training_solutions.json"),\n    "evaluation": ("arc-agi_evaluation-solutions.json", "arc-agi_evaluation_solutions.json"),\n}\n\n\ndef resolve_challenge_path(data_dir: str | Path, split: str) -> Path:\n    return _first_existing(Path(data_dir), CHALLENGE_FILES[split])\n\n\ndef resolve_solution_path(data_dir: str | Path, split: str) -> Path | None:\n    candidates = SOLUTION_FILES.get(split)\n    if not candidates:\n        return None\n    try:\n        return _first_existing(Path(data_dir), candidates)\n    except FileNotFoundError:\n        return None\n\n\ndef _first_existing(data_dir: Path, names: tuple[str, ...]) -> Path:\n    for name in names:\n        path = data_dir / name\n        if path.exists():\n            return path\n    joined = ", ".join(names)\n    raise FileNotFoundError(f"none of these files exist in {data_dir}: {joined}")\n\n\ndef solve_with_fallback(solver, fallback_solver, task: ArcTask) -> Prediction:\n    """Solve one task, degrading to guaranteed-output fallbacks on any failure.\n\n    A Kaggle rerun must always write a submission.json even if some tasks\n    crash their primary solver (model errors, OOM, malformed grids, etc.);\n    letting one bad task abort the whole loop would zero out every other\n    already-solved task too.\n    """\n    try:\n        return solver.solve(task)\n    except Exception as exc:  # noqa: BLE001 - any solver failure must not abort the run\n        print(f"WARNING: {task.id} failed with {solver.__class__.__name__}: {exc!r}; using baseline fallback", file=sys.stderr)\n    try:\n        return fallback_solver.solve(task)\n    except Exception as exc:  # noqa: BLE001 - last-resort guarantee of a valid prediction\n        print(f"WARNING: {task.id} baseline fallback also failed: {exc!r}; using trivial prediction", file=sys.stderr)\n    attempts = [(copy_grid(example.input), [[0]]) for example in task.test]\n    return make_prediction(task, attempts)\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Run Mythos against Kaggle ARC-AGI data.")\n    parser.add_argument("--data-dir", default=str(DEFAULT_KAGGLE_DATA_DIR))\n    parser.add_argument("--split", choices=sorted(CHALLENGE_FILES), default="test")\n    parser.add_argument("--challenges", help="Explicit challenge JSON path; overrides --data-dir/--split.")\n    parser.add_argument("--solutions", help="Explicit solutions JSON path for local scoring.")\n    parser.add_argument("--solver", choices=["pipeline", "baseline", "fixture", "hrm"], default="pipeline")\n    parser.add_argument("--model-mode", choices=["fallback", "strict"], default=None)\n    parser.add_argument("--out", default=str(DEFAULT_KAGGLE_OUTPUT))\n    parser.add_argument("--score", action="store_true", help="Score output when solutions are available.")\n    args = parser.parse_args(argv)\n\n    try:\n        challenge_path = Path(args.challenges) if args.challenges else resolve_challenge_path(args.data_dir, args.split)\n        solution_path = Path(args.solutions) if args.solutions else resolve_solution_path(args.data_dir, args.split)\n\n        tasks = load_challenges(challenge_path)\n        solver = make_solver(args.solver, model_mode=args.model_mode)\n        fallback_solver = solver if isinstance(solver, BaselineSolver) else BaselineSolver()\n        if args.solver == "hrm":\n            try:\n                env = HRMEnvironment.from_env()\n                env.validate(require_cuda=True)\n                if os.environ.get("MYTHOS_ENABLE_TTT") == "1":\n                    runner = HRMTTTRunner(\n                        env,\n                        ttt=TTTConfig(\n                            rank=int(os.environ.get("MYTHOS_TTT_RANK", "16")),\n                            steps=int(os.environ.get("MYTHOS_TTT_STEPS", "20")),\n                            lr=float(os.environ.get("MYTHOS_TTT_LR", "1e-3")),\n                            batch_size=int(os.environ.get("MYTHOS_TTT_BATCH_SIZE", "2")),\n                            genie_weight=float(os.environ.get("MYTHOS_TTT_GENIE_WEIGHT", "0.1")),\n                        ),\n                        num_aug=int(os.environ.get("MYTHOS_TTT_NUM_AUG", "0")),\n                    )\n                else:\n                    runner = HRMInferenceRunner(env)\n                predictions = runner.solve_tasks(list(tasks.values()))\n            except Exception as exc:  # noqa: BLE001 - HRM batch failure must not abort the run\n                print(f"WARNING: HRM batch run failed: {exc!r}; using baseline fallback for all tasks", file=sys.stderr)\n                traceback.print_exc(file=sys.stderr)\n                if getattr(exc, "stdout", None):\n                    print("--- subprocess stdout (tail) ---", file=sys.stderr)\n                    print(exc.stdout[-4000:], file=sys.stderr)\n                if getattr(exc, "stderr", None):\n                    print("--- subprocess stderr (tail) ---", file=sys.stderr)\n                    print(exc.stderr[-4000:], file=sys.stderr)\n                predictions = [fallback_solver.solve(task) for task in tasks.values()]\n        else:\n            predictions = [solve_with_fallback(solver, fallback_solver, task) for task in tasks.values()]\n        write_submission(predictions, args.out)\n\n        summary: dict[str, object] = {\n            "challenge_path": str(challenge_path),\n            "output_path": str(args.out),\n            "solver": args.solver,\n            "model_mode": args.model_mode or "fallback",\n            "tasks": len(tasks),\n        }\n        if hasattr(solver, "pipeline"):\n            summary["models"] = solver.pipeline.model_registry.summary()\n        if args.score and solution_path is not None:\n            summary["score"] = score_files(args.out, str(solution_path)).to_dict()\n        elif args.score:\n            summary["score"] = "skipped: no solutions file for this split"\n    except (ArcValidationError, SolverError, FileNotFoundError, ValueError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n\n    print(json.dumps(summary, indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/lora.py': '"""Minimal LoRA utilities for HRM/TTT adapter experiments."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport importlib\nimport math\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\n\ntry:  # Keep this module importable until a LoRA function is actually used.\n    from torch import nn as _OPTIONAL_NN\nexcept Exception:  # pragma: no cover - depends on optional torch install.\n    _OPTIONAL_NN = None\n\nDEFAULT_LORA_TARGET_PATTERNS = (\n    "attn",\n    "attention",\n    "qkv_proj",\n    "q_proj",\n    "k_proj",\n    "v_proj",\n    "o_proj",\n    "out_proj",\n)\n\n\n@dataclass(frozen=True)\nclass LoRAInjectionReport:\n    injected_modules: tuple[str, ...]\n    trainable_parameters: int\n    frozen_parameters: int\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "injected_modules": list(self.injected_modules),\n            "trainable_parameters": self.trainable_parameters,\n            "frozen_parameters": self.frozen_parameters,\n        }\n\n\ndef _torch():\n    try:\n        import torch\n    except Exception as exc:  # pragma: no cover - depends on optional torch install.\n        raise RuntimeError("PyTorch is required for LoRA utilities") from exc\n    return torch\n\n\ndef _nn():\n    try:\n        from torch import nn\n    except Exception as exc:  # pragma: no cover - depends on optional torch install.\n        raise RuntimeError("PyTorch is required for LoRA utilities") from exc\n    return nn\n\n\n_BASE_MODULE = _OPTIONAL_NN.Module if _OPTIONAL_NN is not None else object\n\n\nclass LoRALinear(_BASE_MODULE):\n    """Wrap a Linear layer with trainable low-rank adapter weights."""\n\n    def __init__(\n        self,\n        base_layer,\n        *,\n        rank: int = 16,\n        alpha: float | None = None,\n        dropout: float = 0.0,\n    ) -> None:\n        nn = _nn()\n        torch = _torch()\n        super().__init__()\n        if not isinstance(base_layer, nn.Linear):\n            raise TypeError("LoRALinear can only wrap torch.nn.Linear")\n        if rank <= 0:\n            raise ValueError("LoRA rank must be positive")\n\n        self.base_layer = base_layer\n        self.rank = rank\n        self.alpha = float(alpha if alpha is not None else rank)\n        self.scaling = self.alpha / float(rank)\n        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()\n        # Match the base layer\'s device/dtype: creating these as plain CPU/fp32\n        # tensors breaks torch.compile\'d models (HRM runs compiled on CUDA in\n        # bfloat16) -- confirmed by a real run: Dynamo\'s tracer rejected the\n        # LoRA matmul with "Unhandled FakeTensor Device Propagation ... found\n        # two different devices cuda:0, cpu".\n        base_device = base_layer.weight.device\n        base_dtype = base_layer.weight.dtype\n        self.lora_a = nn.Parameter(torch.empty(rank, base_layer.in_features, device=base_device, dtype=base_dtype))\n        self.lora_b = nn.Parameter(torch.zeros(base_layer.out_features, rank, device=base_device, dtype=base_dtype))\n        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))\n\n        for parameter in self.base_layer.parameters():\n            parameter.requires_grad = False\n\n    def forward(self, inputs):  # type: ignore[no-untyped-def]\n        torch = _torch()\n        base = self.base_layer(inputs)\n        # CastedLinear (and some plain Linear layers under mixed precision) stores\n        # its weight in one dtype (fp32) but receives activations in another\n        # (bf16) -- cast the activation to lora_a\'s dtype before this matmul, not\n        # base_layer.weight\'s dtype, since those two can legitimately differ.\n        # Confirmed by a real run: "expected mat1 and mat2 to have the same\n        # dtype, but got: c10::BFloat16 != float" once the device mismatch (the\n        # earlier bug) was fixed.\n        update = self.dropout(inputs).to(dtype=self.lora_a.dtype).matmul(self.lora_a.transpose(0, 1))\n        update = update.matmul(self.lora_b.transpose(0, 1))\n        return base + update.to(dtype=base.dtype) * torch.as_tensor(self.scaling, dtype=base.dtype, device=base.device)\n\n\nclass LoRACastedLinear(_BASE_MODULE):\n    """Wrap HRM\'s custom CastedLinear layer with trainable LoRA weights."""\n\n    def __init__(\n        self,\n        base_layer,\n        *,\n        rank: int = 16,\n        alpha: float | None = None,\n        dropout: float = 0.0,\n    ) -> None:\n        nn = _nn()\n        torch = _torch()\n        super().__init__()\n        if not _is_casted_linear_like(base_layer):\n            raise TypeError("LoRACastedLinear can only wrap CastedLinear-like modules")\n        if rank <= 0:\n            raise ValueError("LoRA rank must be positive")\n\n        out_features, in_features = base_layer.weight.shape\n        self.base_layer = base_layer\n        self.rank = rank\n        self.alpha = float(alpha if alpha is not None else rank)\n        self.scaling = self.alpha / float(rank)\n        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()\n        # See LoRALinear\'s identical comment: must match the base layer\'s\n        # device/dtype or torch.compile\'d forward passes fail to trace.\n        base_device = base_layer.weight.device\n        base_dtype = base_layer.weight.dtype\n        self.lora_a = nn.Parameter(torch.empty(rank, in_features, device=base_device, dtype=base_dtype))\n        self.lora_b = nn.Parameter(torch.zeros(out_features, rank, device=base_device, dtype=base_dtype))\n        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))\n\n        for parameter in self.base_layer.parameters():\n            parameter.requires_grad = False\n\n    def forward(self, inputs):  # type: ignore[no-untyped-def]\n        torch = _torch()\n        base = self.base_layer(inputs)\n        # CastedLinear (and some plain Linear layers under mixed precision) stores\n        # its weight in one dtype (fp32) but receives activations in another\n        # (bf16) -- cast the activation to lora_a\'s dtype before this matmul, not\n        # base_layer.weight\'s dtype, since those two can legitimately differ.\n        # Confirmed by a real run: "expected mat1 and mat2 to have the same\n        # dtype, but got: c10::BFloat16 != float" once the device mismatch (the\n        # earlier bug) was fixed.\n        update = self.dropout(inputs).to(dtype=self.lora_a.dtype).matmul(self.lora_a.transpose(0, 1))\n        update = update.matmul(self.lora_b.transpose(0, 1))\n        return base + update.to(dtype=base.dtype) * torch.as_tensor(self.scaling, dtype=base.dtype, device=base.device)\n\n\ndef inject_lora_adapters(\n    model,\n    *,\n    rank: int = 16,\n    alpha: float | None = None,\n    dropout: float = 0.0,\n    target_patterns: Sequence[str] = DEFAULT_LORA_TARGET_PATTERNS,\n    fallback_to_all_linear: bool = False,\n    freeze_backbone: bool = True,\n) -> LoRAInjectionReport:\n    """Replace matching Linear/CastedLinear modules with LoRA wrappers."""\n\n    nn = _nn()\n    lowered_patterns = tuple(pattern.lower() for pattern in target_patterns)\n    injected: list[str] = []\n\n    if freeze_backbone:\n        for parameter in model.parameters():\n            parameter.requires_grad = False\n\n    def should_wrap(full_name: str, child) -> bool:  # type: ignore[no-untyped-def]\n        if not _is_lora_wrappable(child, nn):\n            return False\n        if any(pattern in full_name.lower() for pattern in lowered_patterns):\n            return True\n        return fallback_to_all_linear\n\n    def visit(module, prefix: str = "") -> None:  # type: ignore[no-untyped-def]\n        for child_name, child in list(module.named_children()):\n            full_name = f"{prefix}.{child_name}" if prefix else child_name\n            if should_wrap(full_name, child):\n                setattr(\n                    module,\n                    child_name,\n                    _wrap_lora_layer(child, nn, rank=rank, alpha=alpha, dropout=dropout),\n                )\n                injected.append(full_name)\n            else:\n                visit(child, full_name)\n\n    visit(model)\n    if not injected:\n        raise ValueError("no Linear or CastedLinear modules matched the LoRA target patterns")\n\n    trainable = 0\n    frozen = 0\n    for parameter in model.parameters():\n        if parameter.requires_grad:\n            trainable += parameter.numel()\n        else:\n            frozen += parameter.numel()\n    return LoRAInjectionReport(\n        injected_modules=tuple(injected),\n        trainable_parameters=trainable,\n        frozen_parameters=frozen,\n    )\n\n\ndef lora_parameters(model) -> list:  # type: ignore[no-untyped-def]\n    return [parameter for name, parameter in model.named_parameters() if "lora_" in name and parameter.requires_grad]\n\n\ndef lora_state_dict(model) -> dict[str, object]:  # type: ignore[no-untyped-def]\n    return {\n        name: parameter.detach().cpu()\n        for name, parameter in model.named_parameters()\n        if "lora_" in name\n    }\n\n\ndef save_lora_checkpoint(model, path: str | Path, *, metadata: dict[str, object] | None = None) -> Path:  # type: ignore[no-untyped-def]\n    torch = _torch()\n    output_path = Path(path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(\n        {\n            "kind": "mythos_lora",\n            "state_dict": lora_state_dict(model),\n            "metadata": metadata or {},\n        },\n        output_path,\n    )\n    return output_path\n\n\ndef snapshot_frozen_parameters(model) -> dict[str, object]:  # type: ignore[no-untyped-def]\n    return {\n        name: parameter.detach().clone().cpu()\n        for name, parameter in model.named_parameters()\n        if not parameter.requires_grad\n    }\n\n\ndef changed_frozen_parameters(model, snapshot: dict[str, object], *, atol: float = 0.0) -> tuple[str, ...]:  # type: ignore[no-untyped-def]\n    torch = _torch()\n    changed: list[str] = []\n    for name, before in snapshot.items():\n        current = dict(model.named_parameters())[name].detach().cpu()\n        if not torch.allclose(current, before, atol=atol, rtol=0.0):\n            changed.append(name)\n    return tuple(changed)\n\n\ndef count_parameters(parameters: Iterable) -> int:  # type: ignore[type-arg]\n    return sum(parameter.numel() for parameter in parameters)\n\n\ndef _wrap_lora_layer(child, nn, *, rank: int, alpha: float | None, dropout: float):  # type: ignore[no-untyped-def]\n    if isinstance(child, nn.Linear):\n        return LoRALinear(child, rank=rank, alpha=alpha, dropout=dropout)\n    if _is_casted_linear_like(child):\n        return LoRACastedLinear(child, rank=rank, alpha=alpha, dropout=dropout)\n    raise TypeError(f"unsupported LoRA target module: {type(child).__name__}")\n\n\ndef _is_lora_wrappable(child, nn) -> bool:  # type: ignore[no-untyped-def]\n    return isinstance(child, nn.Linear) or _is_casted_linear_like(child)\n\n\ndef _is_casted_linear_like(child) -> bool:  # type: ignore[no-untyped-def]\n    casted_types = _casted_linear_types()\n    if casted_types and isinstance(child, casted_types):\n        return True\n    if child.__class__.__name__ != "CastedLinear":\n        return False\n    weight = getattr(child, "weight", None)\n    return weight is not None and getattr(weight, "ndim", None) == 2 and callable(getattr(child, "forward", None))\n\n\ndef _casted_linear_types() -> tuple[type, ...]:\n    types: list[type] = []\n    for module_name in ("models.layers", "layers"):\n        try:\n            module = importlib.import_module(module_name)\n        except Exception:\n            continue\n        casted_linear = getattr(module, "CastedLinear", None)\n        if isinstance(casted_linear, type):\n            types.append(casted_linear)\n    return tuple(types)\n', 'src/mythos/losses.py': '"""Loss functions for Mythos adaptation experiments."""\n\nfrom __future__ import annotations\n\nfrom typing import Sequence\n\nfrom mythos.arc import Grid\n\n\ndef genie_background_consistency_loss(\n    logits,\n    input_grid: Grid,\n    *,\n    preserve_mask: Sequence[Sequence[bool]] | None = None,\n):  # type: ignore[no-untyped-def]\n    """Penalize changing cells that should remain visually consistent.\n\n    `logits` may be shaped `[H, W, 10]`, `[1, H, W, 10]`, or `[10, H, W]`.\n    The target color for preserved cells is the original input-grid color.\n    """\n\n    torch = _torch()\n    prepared = _prepare_logits(logits)\n    height = min(prepared.shape[0], len(input_grid))\n    width = min(prepared.shape[1], len(input_grid[0]))\n    mask = preserve_mask or _default_preserve_mask(input_grid)\n\n    selected_logits = []\n    selected_targets = []\n    for row in range(height):\n        for col in range(width):\n            if row < len(mask) and col < len(mask[row]) and mask[row][col]:\n                selected_logits.append(prepared[row, col])\n                selected_targets.append(int(input_grid[row][col]))\n\n    if not selected_logits:\n        return prepared.sum() * 0.0\n\n    logits_tensor = torch.stack(selected_logits, dim=0)\n    target_tensor = torch.tensor(selected_targets, dtype=torch.long, device=prepared.device)\n    return torch.nn.functional.cross_entropy(logits_tensor, target_tensor)\n\n\ndef background_preservation_mask(input_grid: Grid, output_grid: Grid | None = None) -> tuple[tuple[bool, ...], ...]:\n    """Return cells that should be preserved for consistency regularization."""\n\n    if output_grid is not None and _same_shape(input_grid, output_grid):\n        return tuple(\n            tuple(input_grid[row][col] == output_grid[row][col] for col in range(len(input_grid[0])))\n            for row in range(len(input_grid))\n        )\n    return _default_preserve_mask(input_grid)\n\n\ndef _default_preserve_mask(input_grid: Grid) -> tuple[tuple[bool, ...], ...]:\n    background = _dominant_color(input_grid)\n    return tuple(tuple(cell == background for cell in row) for row in input_grid)\n\n\ndef _prepare_logits(logits):  # type: ignore[no-untyped-def]\n    torch = _torch()\n    tensor = logits if hasattr(logits, "shape") else torch.as_tensor(logits)\n    if tensor.ndim == 4:\n        if tensor.shape[0] != 1:\n            raise ValueError("batched consistency loss expects batch size 1")\n        tensor = tensor[0]\n    if tensor.ndim != 3:\n        raise ValueError("logits must have rank 3 or rank 4")\n    if tensor.shape[0] == 10 and tensor.shape[-1] != 10:\n        tensor = tensor.permute(1, 2, 0)\n    if tensor.shape[-1] != 10:\n        raise ValueError("logits must have 10 color channels")\n    return tensor.float()\n\n\ndef _same_shape(left: Grid, right: Grid) -> bool:\n    return len(left) == len(right) and len(left[0]) == len(right[0])\n\n\ndef _dominant_color(grid: Grid) -> int:\n    counts: dict[int, int] = {}\n    for row in grid:\n        for cell in row:\n            counts[cell] = counts.get(cell, 0) + 1\n    return max(counts.items(), key=lambda item: item[1])[0]\n\n\ndef _torch():\n    try:\n        import torch\n    except Exception as exc:  # pragma: no cover - depends on optional torch install.\n        raise RuntimeError("PyTorch is required for Mythos loss functions") from exc\n    return torch\n', 'src/mythos/metrics.py': '"""Scoring helpers for ARC-style submissions."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom typing import Mapping, Tuple\n\nfrom mythos.arc import ArcValidationError, Grid, SolutionMap, grid_equal, load_solutions\nfrom mythos.submission import SubmissionMap, TestPrediction, load_submission\n\n\n@dataclass(frozen=True)\nclass ScoreResult:\n    total_items: int\n    exact_matches: int\n    exact_attempt_1: int\n    exact_attempt_2: int\n    total_cells: int\n    matched_cells: int\n    extra_predictions: int\n\n    @property\n    def exact_accuracy(self) -> float:\n        return self.exact_matches / self.total_items if self.total_items else 0.0\n\n    @property\n    def cell_accuracy(self) -> float:\n        return self.matched_cells / self.total_cells if self.total_cells else 0.0\n\n    def to_dict(self) -> dict[str, int | float]:\n        data = asdict(self)\n        data["exact_accuracy"] = self.exact_accuracy\n        data["cell_accuracy"] = self.cell_accuracy\n        return data\n\n\ndef _cell_count(grid: Grid) -> int:\n    return sum(len(row) for row in grid)\n\n\ndef _cell_matches(prediction: Grid, truth: Grid) -> Tuple[int, int]:\n    total = _cell_count(truth)\n    if len(prediction) != len(truth) or len(prediction[0]) != len(truth[0]):\n        return 0, total\n    matches = 0\n    for pred_row, truth_row in zip(prediction, truth):\n        for pred_cell, truth_cell in zip(pred_row, truth_row):\n            if pred_cell == truth_cell:\n                matches += 1\n    return matches, total\n\n\ndef score_submission_data(predictions: SubmissionMap, solutions: SolutionMap) -> ScoreResult:\n    total_items = 0\n    exact_matches = 0\n    exact_attempt_1 = 0\n    exact_attempt_2 = 0\n    matched_cells = 0\n    total_cells = 0\n\n    extra_predictions = len(set(predictions) - set(solutions))\n\n    for task_id, truth_outputs in solutions.items():\n        if task_id not in predictions:\n            raise ArcValidationError(f"submission is missing task {task_id}")\n        task_predictions = predictions[task_id]\n        if len(task_predictions) != len(truth_outputs):\n            raise ArcValidationError(\n                f"{task_id} has {len(task_predictions)} predictions but "\n                f"{len(truth_outputs)} solution outputs"\n            )\n\n        for prediction, truth in zip(task_predictions, truth_outputs):\n            total_items += 1\n            attempt_1_exact = grid_equal(prediction.attempt_1, truth)\n            attempt_2_exact = grid_equal(prediction.attempt_2, truth)\n            exact_attempt_1 += int(attempt_1_exact)\n            exact_attempt_2 += int(attempt_2_exact)\n            exact_matches += int(attempt_1_exact or attempt_2_exact)\n\n            cells_1, cells_total = _cell_matches(prediction.attempt_1, truth)\n            cells_2, _ = _cell_matches(prediction.attempt_2, truth)\n            matched_cells += max(cells_1, cells_2)\n            total_cells += cells_total\n\n    return ScoreResult(\n        total_items=total_items,\n        exact_matches=exact_matches,\n        exact_attempt_1=exact_attempt_1,\n        exact_attempt_2=exact_attempt_2,\n        total_cells=total_cells,\n        matched_cells=matched_cells,\n        extra_predictions=extra_predictions,\n    )\n\n\ndef score_files(prediction_path: str, solution_path: str) -> ScoreResult:\n    return score_submission_data(load_submission(prediction_path), load_solutions(solution_path))\n', 'src/mythos/models.py': '"""External model loading for the Project Mythos pipeline."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nimport importlib\nimport os\nfrom pathlib import Path\nimport sys\nfrom typing import Any\n\nfrom mythos.solvers.base import SolverError\n\n\nclass ModelLoadError(SolverError):\n    """Raised when a configured external model cannot be loaded."""\n\n\n@dataclass(frozen=True)\nclass ModelSpec:\n    key: str\n    label: str\n    checkpoint_env: str\n    repo_env: str | None = None\n    module_names: tuple[str, ...] = ()\n\n\nMODEL_SPECS: tuple[ModelSpec, ...] = (\n    ModelSpec(\n        key="jepa",\n        label="JEPA encoder",\n        checkpoint_env="IJEPA_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="jepa_projection",\n        label="I-JEPA projection",\n        checkpoint_env="IJEPA_PROJECTION_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="hrm_text",\n        label="HRM-Text H-module",\n        checkpoint_env="HRM_TEXT_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="world_model",\n        label="World model",\n        checkpoint_env="WORLD_MODEL_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="ttt_lora",\n        label="TTT LoRA adapters",\n        checkpoint_env="TTT_LORA_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="hrm_l_module",\n        label="HRM 27M L-module",\n        repo_env="HRM_REPO_DIR",\n        checkpoint_env="HRM_CHECKPOINT_PATH",\n        module_names=("pretrain", "evaluate"),\n    ),\n)\n\n\n@dataclass(frozen=True)\nclass LoadedModel:\n    spec: ModelSpec\n    repo_dir: Path | None = None\n    checkpoint_path: Path | None = None\n    checkpoint: Any = None\n    modules: dict[str, Any] = field(default_factory=dict)\n    error: str | None = None\n\n    @property\n    def loaded(self) -> bool:\n        return self.checkpoint is not None\n\n    def describe(self) -> str:\n        if not self.loaded:\n            if self.error:\n                return f"{self.spec.label}: load failed: {self.error}"\n            return f"{self.spec.label}: not configured"\n        parts = [f"{self.spec.label}: checkpoint={self.checkpoint_path}"]\n        if self.repo_dir is not None:\n            parts.append(f"repo={self.repo_dir}")\n        if self.modules:\n            parts.append(f"modules={\',\'.join(sorted(self.modules))}")\n        return "; ".join(parts)\n\n\nclass ModelRegistry:\n    """Loads and stores external models keyed by planned pipeline component."""\n\n    def __init__(self, models: dict[str, LoadedModel], *, strict: bool) -> None:\n        self.models = models\n        self.strict = strict\n\n    @classmethod\n    def from_env(cls, *, strict: bool = False) -> "ModelRegistry":\n        models: dict[str, LoadedModel] = {}\n        for spec in MODEL_SPECS:\n            try:\n                models[spec.key] = _load_model_from_env(spec, strict=strict)\n            except ModelLoadError as exc:\n                if strict:\n                    raise\n                models[spec.key] = LoadedModel(spec=spec, error=str(exc))\n        return cls(models=models, strict=strict)\n\n    def get(self, key: str) -> LoadedModel:\n        return self.models[key]\n\n    def summary(self) -> list[dict[str, object]]:\n        return [\n            {\n                "key": key,\n                "label": model.spec.label,\n                "loaded": model.loaded,\n                "repo_dir": str(model.repo_dir) if model.repo_dir is not None else None,\n                "checkpoint_path": str(model.checkpoint_path) if model.checkpoint_path is not None else None,\n                "modules": sorted(model.modules),\n                "error": model.error,\n            }\n            for key, model in self.models.items()\n        ]\n\n\ndef _load_model_from_env(spec: ModelSpec, *, strict: bool) -> LoadedModel:\n    repo_value = os.environ.get(spec.repo_env) if spec.repo_env is not None else None\n    checkpoint_value = os.environ.get(spec.checkpoint_env)\n\n    if not repo_value and not checkpoint_value:\n        if strict:\n            missing = spec.checkpoint_env\n            if spec.repo_env is not None:\n                missing = f"{spec.repo_env} and {spec.checkpoint_env}"\n            raise ModelLoadError(f"{missing} are required for strict model loading")\n        return LoadedModel(spec=spec)\n\n    if spec.repo_env is not None and not repo_value:\n        raise ModelLoadError(f"{spec.repo_env} is required when loading {spec.label}")\n    if not checkpoint_value:\n        raise ModelLoadError(f"{spec.checkpoint_env} is required when loading {spec.label}")\n\n    repo_dir = Path(repo_value) if repo_value else None\n    checkpoint_path = Path(checkpoint_value)\n\n    if repo_dir is not None:\n        if not repo_dir.exists():\n            raise ModelLoadError(f"{spec.repo_env} does not exist: {repo_dir}")\n        _add_repo_to_path(repo_dir)\n\n    if not checkpoint_path.exists():\n        raise ModelLoadError(f"{spec.checkpoint_env} does not exist: {checkpoint_path}")\n\n    modules = _import_modules(spec)\n    checkpoint = _load_torch_checkpoint(checkpoint_path)\n    return LoadedModel(\n        spec=spec,\n        repo_dir=repo_dir,\n        checkpoint_path=checkpoint_path,\n        checkpoint=checkpoint,\n        modules=modules,\n    )\n\n\ndef _add_repo_to_path(repo_dir: Path) -> None:\n    repo = str(repo_dir.resolve())\n    if repo not in sys.path:\n        sys.path.insert(0, repo)\n\n\ndef _import_modules(spec: ModelSpec) -> dict[str, Any]:\n    modules: dict[str, Any] = {}\n    for module_name in spec.module_names:\n        try:\n            modules[module_name] = importlib.import_module(module_name)\n        except Exception as exc:\n            raise ModelLoadError(\n                f"failed to import {module_name!r} for {spec.label}: {exc}"\n            ) from exc\n    return modules\n\n\ndef _load_torch_checkpoint(checkpoint_path: Path) -> Any:\n    if checkpoint_path.is_dir() or checkpoint_path.suffix.lower() == ".safetensors":\n        return {\n            "path": str(checkpoint_path),\n            "format": checkpoint_path.suffix.lower().lstrip(".") or "directory",\n            "lazy": True,\n        }\n\n    try:\n        torch = importlib.import_module("torch")\n    except Exception as exc:\n        raise ModelLoadError("PyTorch is required to load model checkpoints") from exc\n\n    map_location = "cuda" if torch.cuda.is_available() else "cpu"\n    try:\n        return torch.load(checkpoint_path, map_location=map_location, weights_only=False)\n    except TypeError:\n        return torch.load(checkpoint_path, map_location=map_location)\n', 'src/mythos/pipeline.py': '"""Plan-aligned Project Mythos inference pipeline.\n\nThe real research components are still adapters here. The important point for\nthe base implementation is that data flows through the same stage boundaries as\nthe master plan, so each placeholder has an obvious replacement point.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass, field\nimport os\nfrom typing import Iterable, Tuple\n\nfrom mythos.arc import ArcTask, Grid, copy_grid\nfrom mythos.features import (\n    DEFAULT_HRM_FEATURE_DIM,\n    DEFAULT_JEPA_FEATURE_DIM,\n    embedding_cosine_similarity,\n    grid_to_feature_vector,\n    task_rule_vector,\n)\nfrom mythos.jepa_encoder import JepaEncodingError, JepaImageEncoder\nfrom mythos.models import ModelRegistry\nfrom mythos.solvers.baseline import BaselineSolver\nfrom mythos.solvers.base import SolverError\nfrom mythos.solvers.hrm import HRMSolver\nfrom mythos.submission import Prediction, TestPrediction, prediction_to_json, validate_submission_data\nfrom mythos.text_reasoning import HRMTextError, generate_hrm_text_rule\nfrom mythos.training import load_projection_checkpoint, load_world_model_checkpoint\n\nPLAN_STAGE_ORDER = (\n    "ingest",\n    "encode_jepa",\n    "plan_hrm_text",\n    "simulate_world_model",\n    "adapt_ttt_lora",\n    "execute_hrm_l_module",\n    "decode_output",\n)\n\n\n@dataclass(frozen=True)\nclass StageRecord:\n    name: str\n    status: str\n    detail: str\n\n\n@dataclass(frozen=True)\nclass GridEmbedding:\n    source: str\n    shape: tuple[int, int]\n    vector: tuple[float, ...]\n\n\n@dataclass(frozen=True)\nclass RuleVector:\n    source: str\n    description: str\n    vector: tuple[float, ...]\n\n\n@dataclass\nclass PipelineTrace:\n    task_id: str\n    stages: list[StageRecord] = field(default_factory=list)\n\n    @property\n    def stage_names(self) -> list[str]:\n        return [stage.name for stage in self.stages]\n\n    def add(self, name: str, status: str, detail: str) -> None:\n        self.stages.append(StageRecord(name=name, status=status, detail=detail))\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "task_id": self.task_id,\n            "stages": [\n                {"name": stage.name, "status": stage.status, "detail": stage.detail}\n                for stage in self.stages\n            ],\n        }\n\n\n@dataclass(frozen=True)\nclass PipelineResult:\n    prediction: Prediction\n    trace: PipelineTrace\n\n\n@dataclass\nclass PipelineState:\n    task: ArcTask\n    trace: PipelineTrace\n    embeddings: tuple[GridEmbedding, ...] = ()\n    rule_vector: RuleVector | None = None\n    world_rollouts: tuple[GridEmbedding, ...] = ()\n    prediction: Prediction | None = None\n\n\nclass PlannedPipeline:\n    """Runs the ARC task through the Project Mythos master-plan stages."""\n\n    def __init__(\n        self,\n        executor: BaselineSolver | None = None,\n        model_registry: ModelRegistry | None = None,\n        *,\n        strict_models: bool = False,\n    ) -> None:\n        self.executor = executor or BaselineSolver()\n        self.model_registry = model_registry or ModelRegistry.from_env(strict=strict_models)\n        self.strict_models = strict_models\n        self._jepa_encoder = None\n        self._projection_model = None\n        self._world_model = None\n\n    def run(self, task: ArcTask) -> PipelineResult:\n        state = PipelineState(task=task, trace=PipelineTrace(task_id=task.id))\n\n        self._ingest(state)\n        self._encode_jepa(state)\n        self._plan_hrm_text(state)\n        self._simulate_world_model(state)\n        self._adapt_ttt_lora(state)\n        self._execute_hrm_l_module(state)\n        self._decode_output(state)\n\n        assert state.prediction is not None\n        return PipelineResult(prediction=state.prediction, trace=state.trace)\n\n    def _ingest(self, state: PipelineState) -> None:\n        train_count = len(state.task.train)\n        test_count = len(state.task.test)\n        state.trace.add(\n            "ingest",\n            "ok",\n            f"loaded task with {train_count} train pairs and {test_count} test inputs",\n        )\n\n    def _encode_jepa(self, state: PipelineState) -> None:\n        jepa_model = self.model_registry.get("jepa")\n        projection = self.model_registry.get("jepa_projection")\n        embeddings: list[GridEmbedding] = []\n        projection_detail = "projection checkpoint not configured"\n        projection_loaded = False\n        real_jepa_enabled = os.environ.get("MYTHOS_ENABLE_REAL_JEPA", "0") == "1"\n        real_jepa_loaded = False\n        try:\n            jepa_encoder = None\n            if real_jepa_enabled and jepa_model.loaded and jepa_model.checkpoint_path is not None:\n                jepa_encoder = self._get_jepa_encoder(jepa_model.checkpoint_path)\n                real_jepa_loaded = True\n            if projection.loaded and projection.checkpoint_path is not None:\n                projection_model = self._get_projection_model(projection.checkpoint_path)\n                projection_loaded = True\n                projection_detail = f"projection={projection.checkpoint_path}"\n                for split, grids in _iter_task_grids(state.task):\n                    for index, grid in enumerate(grids):\n                        embeddings.append(\n                            _encode_jepa_grid(\n                                f"{split}[{index}]",\n                                grid,\n                                jepa_encoder=jepa_encoder,\n                                projection_model=projection_model,\n                            )\n                        )\n            else:\n                for split, grids in _iter_task_grids(state.task):\n                    for index, grid in enumerate(grids):\n                        embeddings.append(\n                            _encode_jepa_grid(\n                                f"{split}[{index}]",\n                                grid,\n                                jepa_encoder=jepa_encoder,\n                                projection_model=None,\n                            )\n                        )\n        except Exception as exc:\n            if self.strict_models:\n                raise\n            projection_detail = f"projection failed: {exc}; deterministic ARC-feature fallback used"\n            embeddings = [\n                _feature_grid(f"{split}[{index}]", grid, dim=DEFAULT_JEPA_FEATURE_DIM)\n                for split, grids in _iter_task_grids(state.task)\n                for index, grid in enumerate(grids)\n            ]\n        state.embeddings = tuple(embeddings)\n        if real_jepa_loaded and projection_loaded:\n            status = "jepa_forward_projection_loaded"\n        elif real_jepa_loaded:\n            status = "jepa_forward"\n        elif projection_loaded:\n            status = "projection_loaded"\n        else:\n            status = "fallback"\n\n        if real_jepa_loaded:\n            detail = (\n                f"{jepa_model.describe()}; {projection.describe()}; "\n                f"MYTHOS_ENABLE_REAL_JEPA=1; transformers I-JEPA forward ran; "\n                f"{projection_detail}; produced {len(embeddings)} embeddings"\n            )\n        elif projection_loaded:\n            detail = (\n                f"{jepa_model.describe()}; {projection.describe()}; {projection_detail}; "\n                "no I-JEPA forward pass is run without MYTHOS_ENABLE_REAL_JEPA=1; "\n                f"projected {len(embeddings)} deterministic ARC-feature embeddings"\n            )\n        else:\n            detail = (\n                f"{jepa_model.describe()}; {projection.describe()}; {projection_detail}; "\n                f"MYTHOS_ENABLE_REAL_JEPA={int(real_jepa_enabled)}; no I-JEPA forward pass ran; produced "\n                f"{len(embeddings)} deterministic ARC-feature embeddings"\n            )\n        state.trace.add(\n            "encode_jepa",\n            status,\n            detail,\n        )\n\n    def _plan_hrm_text(self, state: PipelineState) -> None:\n        model = self.model_registry.get("hrm_text")\n        enabled = os.environ.get("MYTHOS_ENABLE_HRM_TEXT", "0") == "1"\n        state.rule_vector = _make_rule_vector(state.task)\n        status = "fallback"\n        detail = (\n            f"{model.describe()}; MYTHOS_ENABLE_HRM_TEXT={int(enabled)}; "\n            "deterministic fallback rule vector derived from train-pair deltas"\n        )\n        if enabled and model.loaded and model.checkpoint_path is not None:\n            try:\n                result = generate_hrm_text_rule(state.task, model.checkpoint_path)\n                state.rule_vector = RuleVector(\n                    source="hrm_text_forward",\n                    description=result.description,\n                    vector=result.vector,\n                )\n                status = "model_forward"\n                detail = (\n                    f"{model.describe()}; HRM-Text forward/generation ran from "\n                    f"{result.model_root}; rule={result.description[:160]!r}"\n                )\n            except HRMTextError as exc:\n                if self.strict_models:\n                    raise\n                status = "fallback"\n                detail = (\n                    f"{model.describe()}; HRM-Text forward failed: {exc}; "\n                    "deterministic fallback rule vector used"\n                )\n        elif model.loaded and not enabled:\n            status = "disabled_checkpoint_loaded"\n        state.trace.add(\n            "plan_hrm_text",\n            status,\n            detail,\n        )\n\n    def _simulate_world_model(self, state: PipelineState) -> None:\n        if state.rule_vector is None:\n            raise RuntimeError("rule vector must exist before world-model simulation")\n        model = self.model_registry.get("world_model")\n        status = "model_loaded" if model.loaded else "fallback"\n        detail = f"{model.describe()}; no world-model checkpoint, so no rollout guidance was produced"\n        if model.loaded and model.checkpoint_path is not None:\n            try:\n                world_model = self._get_world_model(model.checkpoint_path)\n                state.world_rollouts = tuple(_simulate_rollouts(state.task, world_model))\n                detail = (\n                    f"{model.describe()}; generated {len(state.world_rollouts)} "\n                    "test-input latent rollouts"\n                )\n            except Exception as exc:\n                if self.strict_models:\n                    raise\n                status = "fallback"\n                detail = f"{model.describe()}; world-model rollout failed: {exc}"\n        state.trace.add(\n            "simulate_world_model",\n            status,\n            detail,\n        )\n\n    def _adapt_ttt_lora(self, state: PipelineState) -> None:\n        model = self.model_registry.get("ttt_lora")\n        hrm = self.model_registry.get("hrm_l_module")\n        enabled = os.environ.get("MYTHOS_ENABLE_TTT", "0") == "1"\n        status = "model_loaded" if model.loaded else "fallback"\n        if enabled and hrm.loaded:\n            detail = (\n                f"{model.describe()}; MYTHOS_ENABLE_TTT=1, but live LoRA-on-HRM "\n                "is not connected because the external HRM train-step hook is not "\n                "wrapped by this pipeline yet"\n            )\n            status = "not_connected"\n        else:\n            detail = (\n                f"{model.describe()}; per-task LoRA update skipped "\n                f"(MYTHOS_ENABLE_TTT={int(enabled)}, hrm_loaded={hrm.loaded})"\n            )\n        state.trace.add(\n            "adapt_ttt_lora",\n            status,\n            detail,\n        )\n\n    def _execute_hrm_l_module(self, state: PipelineState) -> None:\n        model = self.model_registry.get("hrm_l_module")\n        enable_real_hrm = os.environ.get("MYTHOS_ENABLE_REAL_HRM", "0") == "1"\n        status = "model_loaded_fallback_executor" if model.loaded else "fallback"\n        if enable_real_hrm:\n            try:\n                state.prediction = HRMSolver().solve(state.task)\n                status = "model_loaded"\n                detail = f"{model.describe()}; external HRM prediction path returned output"\n            except SolverError as exc:\n                if self.strict_models:\n                    raise\n                state.prediction = self.executor.solve(state.task)\n                status = "fallback"\n                detail = f"{model.describe()}; HRM execution failed: {exc}; baseline executor used"\n        else:\n            state.prediction = self.executor.solve(state.task)\n            detail = (\n                f"{model.describe()}; baseline executor produced valid output "\n                "because MYTHOS_ENABLE_REAL_HRM is not set"\n            )\n        if state.prediction is not None and state.world_rollouts:\n            state.prediction, guided_count = _apply_world_rollout_attempts(\n                state.task,\n                state.prediction,\n                state.world_rollouts,\n            )\n            detail += f"; world-model rollout guidance replaced attempt_2 for {guided_count} test item(s)"\n        state.trace.add(\n            "execute_hrm_l_module",\n            status,\n            detail,\n        )\n\n    def _decode_output(self, state: PipelineState) -> None:\n        if state.prediction is None:\n            raise RuntimeError("prediction must exist before decode/output")\n        validate_submission_data({state.prediction.task_id: prediction_to_json(state.prediction)})\n        state.trace.add(\n            "decode_output",\n            "ok",\n            f"validated {len(state.prediction.outputs)} two-attempt test predictions",\n        )\n\n    def _get_projection_model(self, checkpoint_path):  # type: ignore[no-untyped-def]\n        if self._projection_model is None:\n            self._projection_model = load_projection_checkpoint(checkpoint_path)\n        return self._projection_model\n\n    def _get_jepa_encoder(self, checkpoint_path):  # type: ignore[no-untyped-def]\n        if self._jepa_encoder is None:\n            self._jepa_encoder = JepaImageEncoder.from_path(checkpoint_path)\n        return self._jepa_encoder\n\n    def _get_world_model(self, checkpoint_path):  # type: ignore[no-untyped-def]\n        if self._world_model is None:\n            self._world_model = load_world_model_checkpoint(checkpoint_path)\n        return self._world_model\n\n\ndef _iter_task_grids(task: ArcTask) -> Iterable[tuple[str, tuple[Grid, ...]]]:\n    yield "train_input", tuple(example.input for example in task.train)\n    yield "train_output", tuple(example.output for example in task.train if example.output is not None)\n    yield "test_input", tuple(example.input for example in task.test)\n\n\ndef _encode_grid(source: str, grid: Grid) -> GridEmbedding:\n    height = len(grid)\n    width = len(grid[0])\n    flat = [cell for row in grid for cell in row]\n    counts = Counter(flat)\n    dominant_color = counts.most_common(1)[0][0]\n    nonzero = sum(1 for cell in flat if cell != 0)\n    total = len(flat)\n    vector = (\n        height / 30.0,\n        width / 30.0,\n        dominant_color / 9.0,\n        nonzero / total,\n        sum(flat) / (9.0 * total),\n    )\n    return GridEmbedding(source=source, shape=(height, width), vector=_rounded(vector))\n\n\ndef _feature_grid(source: str, grid: Grid, *, dim: int) -> GridEmbedding:\n    return GridEmbedding(\n        source=source,\n        shape=(len(grid), len(grid[0])),\n        vector=grid_to_feature_vector(grid, dim),\n    )\n\n\ndef _encode_jepa_grid(\n    source: str,\n    grid: Grid,\n    *,\n    jepa_encoder,\n    projection_model,\n) -> GridEmbedding:  # type: ignore[no-untyped-def]\n    if jepa_encoder is not None:\n        vector = jepa_encoder.encode_grid(grid)\n        vector_source = "i_jepa_forward"\n    else:\n        feature_dim = projection_model.config.input_dim if projection_model is not None else DEFAULT_JEPA_FEATURE_DIM\n        vector = grid_to_feature_vector(grid, feature_dim)\n        vector_source = "deterministic_arc_features"\n\n    if projection_model is not None:\n        vector = _project_vector(vector, projection_model)\n        vector_source += "_projected"\n\n    return GridEmbedding(\n        source=f"{source}.{vector_source}",\n        shape=(len(grid), len(grid[0])),\n        vector=_rounded(float(value) for value in vector),\n    )\n\n\ndef _project_grid(source: str, grid: Grid, projection_model) -> GridEmbedding:  # type: ignore[no-untyped-def]\n    vector = grid_to_feature_vector(grid, projection_model.config.input_dim)\n    return GridEmbedding(\n        source=source,\n        shape=(len(grid), len(grid[0])),\n        vector=_project_vector(vector, projection_model),\n    )\n\n\ndef _project_vector(vector: tuple[float, ...], projection_model) -> tuple[float, ...]:  # type: ignore[no-untyped-def]\n    import torch\n\n    if len(vector) != projection_model.config.input_dim:\n        raise JepaEncodingError(\n            f"projection expects {projection_model.config.input_dim} features, got {len(vector)}"\n        )\n    device = next(projection_model.parameters()).device\n    tensor = torch.tensor([vector], dtype=torch.float32, device=device)\n    with torch.no_grad():\n        projected = projection_model(tensor)[0].detach().cpu().tolist()\n    return _rounded(float(value) for value in projected)\n\n\ndef _simulate_rollouts(task: ArcTask, world_model) -> Iterable[GridEmbedding]:  # type: ignore[no-untyped-def]\n    import torch\n\n    device = next(world_model.parameters()).device\n    rule = torch.tensor(\n        [task_rule_vector(task, world_model.config.rule_dim)],\n        dtype=torch.float32,\n        device=device,\n    )\n    for index, example in enumerate(task.test):\n        z_input = torch.tensor(\n            [grid_to_feature_vector(example.input, world_model.config.z_dim)],\n            dtype=torch.float32,\n            device=device,\n        )\n        with torch.no_grad():\n            rollout = world_model(z_input, rule)[0].detach().cpu().tolist()\n        yield GridEmbedding(\n            source=f"test_input[{index}].world_rollout",\n            shape=(len(example.input), len(example.input[0])),\n            vector=_rounded(float(value) for value in rollout),\n        )\n\n\ndef _apply_world_rollout_attempts(\n    task: ArcTask,\n    prediction: Prediction,\n    rollouts: tuple[GridEmbedding, ...],\n) -> tuple[Prediction, int]:\n    train_outputs = [\n        (grid_to_feature_vector(example.output, len(rollouts[0].vector)), example.output)\n        for example in task.train\n        if example.output is not None\n    ]\n    if not train_outputs:\n        return prediction, 0\n\n    guided_outputs: list[TestPrediction] = []\n    guided_count = 0\n    for index, item in enumerate(prediction.outputs):\n        if index >= len(rollouts):\n            guided_outputs.append(item)\n            continue\n        rollout = rollouts[index]\n        nearest_grid = max(\n            train_outputs,\n            key=lambda candidate: embedding_cosine_similarity(rollout.vector, candidate[0]),\n        )[1]\n        guided_outputs.append(\n            TestPrediction(\n                attempt_1=item.attempt_1,\n                attempt_2=copy_grid(nearest_grid),\n            )\n        )\n        guided_count += 1\n    return Prediction(task_id=prediction.task_id, outputs=tuple(guided_outputs)), guided_count\n\n\ndef _make_rule_vector(task: ArcTask) -> RuleVector:\n    vector = task_rule_vector(task)\n    return RuleVector(\n        source="deterministic_train_pair_delta",\n        description="shape, density, and color-overlap summary from demonstration pairs",\n        vector=vector,\n    )\n\n\ndef _nonzero_count(grid: Grid) -> int:\n    return sum(1 for row in grid for cell in row if cell != 0)\n\n\ndef _color_jaccard(left: Grid, right: Grid) -> float:\n    left_colors = {cell for row in left for cell in row}\n    right_colors = {cell for row in right for cell in row}\n    union = left_colors | right_colors\n    if not union:\n        return 1.0\n    return len(left_colors & right_colors) / len(union)\n\n\ndef _average(values: Iterable[float]) -> float:\n    collected = list(values)\n    return sum(collected) / len(collected) if collected else 0.0\n\n\ndef _rounded(values: Iterable[float]) -> Tuple[float, ...]:\n    return tuple(round(value, 6) for value in values)\n', 'src/mythos/score.py': '"""CLI for scoring a submission against solution JSON."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\n\nfrom mythos.arc import ArcValidationError\nfrom mythos.metrics import score_files\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Score an ARC submission JSON file.")\n    parser.add_argument("--pred", required=True, help="Path to submission JSON.")\n    parser.add_argument("--solutions", required=True, help="Path to solution JSON.")\n    args = parser.parse_args(argv)\n\n    try:\n        result = score_files(args.pred, args.solutions)\n    except ArcValidationError as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n    print(json.dumps(result.to_dict(), indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/solve.py': '"""CLI for running a solver and writing submission JSON."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\n\nfrom mythos.arc import ArcValidationError, load_challenges\nfrom mythos.solvers.base import SolverError\nfrom mythos.solvers.factory import make_solver\nfrom mythos.submission import write_submission\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Run a Mythos solver.")\n    parser.add_argument("--solver", choices=["pipeline", "baseline", "fixture", "hrm"], default="pipeline")\n    parser.add_argument("--model-mode", choices=["fallback", "strict"], default=None)\n    parser.add_argument("--challenges", required=True, help="Path to ARC-style challenges JSON.")\n    parser.add_argument("--out", required=True, help="Output submission JSON path.")\n    args = parser.parse_args(argv)\n\n    try:\n        tasks = load_challenges(args.challenges)\n        solver = make_solver(args.solver, model_mode=args.model_mode)\n        predictions = [solver.solve(task) for task in tasks.values()]\n        write_submission(predictions, args.out)\n    except (ArcValidationError, SolverError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n    print(f"Wrote {len(predictions)} predictions to {args.out}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/solvers/__init__.py': '"""Solver implementations."""\n\nfrom mythos.solvers.baseline import BaselineSolver\nfrom mythos.solvers.base import Solver, SolverError\nfrom mythos.solvers.fixture import FixtureSolver\nfrom mythos.solvers.hrm import HRMEnvironmentError, HRMSolver\nfrom mythos.solvers.pipeline import PlannedPipelineSolver\n\n__all__ = [\n    "BaselineSolver",\n    "FixtureSolver",\n    "HRMEnvironmentError",\n    "HRMSolver",\n    "PlannedPipelineSolver",\n    "Solver",\n    "SolverError",\n]\n', 'src/mythos/solvers/base.py': '"""Common solver types."""\n\nfrom __future__ import annotations\n\nfrom typing import Protocol\n\nfrom mythos.arc import ArcTask, Grid\nfrom mythos.submission import Prediction, TestPrediction\n\n\nclass SolverError(RuntimeError):\n    """Raised when a solver cannot produce a prediction."""\n\n\nclass Solver(Protocol):\n    def solve(self, task: ArcTask) -> Prediction:\n        """Return a two-attempt prediction for every test item in a task."""\n\n\ndef make_prediction(task: ArcTask, attempts: list[tuple[Grid, Grid]]) -> Prediction:\n    if len(attempts) != len(task.test):\n        raise SolverError(\n            f"{task.id}: expected {len(task.test)} test predictions, got {len(attempts)}"\n        )\n    return Prediction(\n        task_id=task.id,\n        outputs=tuple(TestPrediction(attempt_1=a1, attempt_2=a2) for a1, a2 in attempts),\n    )\n', 'src/mythos/solvers/baseline.py': '"""Guaranteed-output baseline solver for smoke runs and Kaggle plumbing tests."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\n\nfrom mythos.arc import ArcTask, Grid, copy_grid\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.solvers.fixture import FixtureSolver\nfrom mythos.submission import Prediction\n\n\nclass BaselineSolver:\n    """Try simple fixture rules, then fall back to valid low-skill predictions."""\n\n    def __init__(self) -> None:\n        self.fixture_solver = FixtureSolver()\n\n    def solve(self, task: ArcTask) -> Prediction:\n        try:\n            return self.fixture_solver.solve(task)\n        except SolverError:\n            attempts = [\n                (copy_grid(example.input), _blank_output_for_task(task, example.input))\n                for example in task.test\n            ]\n            return make_prediction(task, attempts)\n\n\ndef _blank_output_for_task(task: ArcTask, input_grid: Grid) -> Grid:\n    height, width = _fallback_shape(task, input_grid)\n    color = _dominant_output_color(task)\n    return [[color for _ in range(width)] for _ in range(height)]\n\n\ndef _fallback_shape(task: ArcTask, input_grid: Grid) -> tuple[int, int]:\n    output_shapes = {\n        (len(example.output), len(example.output[0]))\n        for example in task.train\n        if example.output is not None\n    }\n    if len(output_shapes) == 1:\n        return next(iter(output_shapes))\n    return len(input_grid), len(input_grid[0])\n\n\ndef _dominant_output_color(task: ArcTask) -> int:\n    counts: Counter[int] = Counter()\n    for example in task.train:\n        if example.output is None:\n            continue\n        for row in example.output:\n            counts.update(row)\n    if not counts:\n        return 0\n    return counts.most_common(1)[0][0]\n', 'src/mythos/solvers/factory.py': '"""Solver factory shared by CLIs."""\n\nfrom __future__ import annotations\n\nimport os\n\nfrom mythos.solvers.baseline import BaselineSolver\nfrom mythos.solvers.fixture import FixtureSolver\nfrom mythos.solvers.hrm import HRMSolver\nfrom mythos.solvers.pipeline import PlannedPipelineSolver\n\n\ndef make_solver(name: str, *, model_mode: str | None = None):\n    selected_mode = model_mode or os.environ.get("MYTHOS_MODEL_MODE", "fallback")\n    if selected_mode not in {"fallback", "strict"}:\n        raise ValueError(f"unknown model mode: {selected_mode}")\n    strict_models = selected_mode == "strict"\n    if name == "pipeline":\n        return PlannedPipelineSolver(strict_models=strict_models)\n    if name == "baseline":\n        return BaselineSolver()\n    if name == "fixture":\n        return FixtureSolver()\n    if name == "hrm":\n        return HRMSolver()\n    raise ValueError(f"unknown solver: {name}")\n', 'src/mythos/solvers/fixture.py': '"""Small deterministic solver for the committed toy fixtures."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Callable, Iterable, List, Optional, Tuple\n\nfrom mythos.arc import ArcTask, Grid, copy_grid, grid_equal\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.submission import Prediction\n\nTransform = Callable[[Grid], Grid]\n\n\n@dataclass(frozen=True)\nclass Candidate:\n    name: str\n    transform: Transform\n\n\nclass FixtureSolver:\n    """Infer one simple transformation from train examples and apply it to tests."""\n\n    def solve(self, task: ArcTask) -> Prediction:\n        candidates = _matching_candidates(task)\n        if not candidates:\n            raise SolverError(f"{task.id}: no fixture transformation matched train examples")\n\n        primary = candidates[0].transform\n        secondary = candidates[1].transform if len(candidates) > 1 else primary\n        attempts = [(primary(example.input), secondary(example.input)) for example in task.test]\n        return make_prediction(task, attempts)\n\n\ndef _matching_candidates(task: ArcTask) -> List[Candidate]:\n    candidates = _base_candidates()\n    candidates.extend(_translation_candidates(task))\n    recolor = _recolor_candidate(task)\n    if recolor is not None:\n        candidates.append(recolor)\n\n    matched: List[Candidate] = []\n    seen_outputs: set[str] = set()\n    for candidate in candidates:\n        if _fits(task, candidate.transform):\n            signature = _candidate_signature(task, candidate.transform)\n            if signature not in seen_outputs:\n                matched.append(candidate)\n                seen_outputs.add(signature)\n    return matched\n\n\ndef _candidate_signature(task: ArcTask, transform: Transform) -> str:\n    return repr([transform(example.input) for example in task.test])\n\n\ndef _fits(task: ArcTask, transform: Transform) -> bool:\n    for example in task.train:\n        if example.output is None:\n            return False\n        try:\n            predicted = transform(example.input)\n        except ValueError:\n            return False\n        if not grid_equal(predicted, example.output):\n            return False\n    return True\n\n\ndef _base_candidates() -> List[Candidate]:\n    return [\n        Candidate("identity", copy_grid),\n        Candidate("mirror_horizontal", _mirror_horizontal),\n        Candidate("mirror_vertical", _mirror_vertical),\n        Candidate("rotate_clockwise", _rotate_clockwise),\n        Candidate("rotate_180", lambda grid: _rotate_clockwise(_rotate_clockwise(grid))),\n        Candidate("rotate_counterclockwise", _rotate_counterclockwise),\n    ]\n\n\ndef _mirror_horizontal(grid: Grid) -> Grid:\n    return [list(reversed(row)) for row in grid]\n\n\ndef _mirror_vertical(grid: Grid) -> Grid:\n    return [row[:] for row in reversed(grid)]\n\n\ndef _rotate_clockwise(grid: Grid) -> Grid:\n    return [list(row) for row in zip(*grid[::-1])]\n\n\ndef _rotate_counterclockwise(grid: Grid) -> Grid:\n    return [list(row) for row in zip(*grid)][::-1]\n\n\ndef _recolor_candidate(task: ArcTask) -> Optional[Candidate]:\n    mapping: dict[int, int] = {}\n    for example in task.train:\n        if example.output is None:\n            return None\n        if len(example.input) != len(example.output) or len(example.input[0]) != len(example.output[0]):\n            return None\n        for in_row, out_row in zip(example.input, example.output):\n            for in_cell, out_cell in zip(in_row, out_row):\n                previous = mapping.setdefault(in_cell, out_cell)\n                if previous != out_cell:\n                    return None\n\n    def transform(grid: Grid) -> Grid:\n        return [[mapping.get(cell, cell) for cell in row] for row in grid]\n\n    return Candidate("recolor", transform)\n\n\ndef _translation_candidates(task: ArcTask) -> List[Candidate]:\n    offsets: Optional[set[tuple[int, int]]] = None\n    for example in task.train:\n        if example.output is None:\n            return []\n        example_offsets = set(_valid_translation_offsets(example.input, example.output))\n        offsets = example_offsets if offsets is None else offsets & example_offsets\n    return [\n        Candidate(f"translate_{dr}_{dc}", _translate_transform(dr, dc))\n        for dr, dc in sorted(offsets or set())\n        if dr != 0 or dc != 0\n    ]\n\n\ndef _valid_translation_offsets(source: Grid, target: Grid) -> Iterable[tuple[int, int]]:\n    if len(source) != len(target) or len(source[0]) != len(target[0]):\n        return []\n\n    source_cells = _foreground_cells(source)\n    target_cells = _foreground_cells(target)\n    if len(source_cells) != len(target_cells):\n        return []\n    if not source_cells and not target_cells:\n        return [(0, 0)]\n\n    offsets = []\n    first_r, first_c, first_value = source_cells[0]\n    for target_r, target_c, target_value in target_cells:\n        if target_value != first_value:\n            continue\n        dr = target_r - first_r\n        dc = target_c - first_c\n        try:\n            translated = _translate_grid(source, dr, dc)\n        except ValueError:\n            continue\n        if translated == target:\n            offsets.append((dr, dc))\n    return offsets\n\n\ndef _foreground_cells(grid: Grid) -> List[tuple[int, int, int]]:\n    return [\n        (row_idx, col_idx, cell)\n        for row_idx, row in enumerate(grid)\n        for col_idx, cell in enumerate(row)\n        if cell != 0\n    ]\n\n\ndef _translate_transform(dr: int, dc: int) -> Transform:\n    def transform(grid: Grid) -> Grid:\n        return _translate_grid(grid, dr, dc)\n\n    return transform\n\n\ndef _translate_grid(grid: Grid, dr: int, dc: int) -> Grid:\n    height = len(grid)\n    width = len(grid[0])\n    translated = [[0 for _ in range(width)] for _ in range(height)]\n    for row_idx, row in enumerate(grid):\n        for col_idx, cell in enumerate(row):\n            if cell == 0:\n                continue\n            next_r = row_idx + dr\n            next_c = col_idx + dc\n            if next_r < 0 or next_r >= height or next_c < 0 or next_c >= width:\n                raise ValueError("translation moves cell out of bounds")\n            translated[next_r][next_c] = cell\n    return translated\n', 'src/mythos/solvers/hrm.py': '"""External HRM adapter and environment checks."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport importlib\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nfrom typing import Any\n\nfrom mythos.arc import ArcTask\nfrom mythos.features import ARC_MAX_SIZE, hrm_sequence_to_grid, output_shape_hint\nfrom mythos.hrm_dataset import build_hrm_dataset, default_run_dir, prepare_hrm_raw_dataset\nfrom mythos.losses import genie_background_consistency_loss\nfrom mythos.lora import inject_lora_adapters, lora_parameters\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.submission import Prediction\n\n\nclass HRMEnvironmentError(SolverError):\n    """Raised when the external HRM runtime is not ready."""\n\n\n@dataclass(frozen=True)\nclass HRMEnvironment:\n    repo_dir: Path\n    checkpoint_path: Path\n\n    @classmethod\n    def from_env(cls) -> "HRMEnvironment":\n        repo_value = os.environ.get("HRM_REPO_DIR")\n        checkpoint_value = os.environ.get("HRM_CHECKPOINT_PATH")\n        if not repo_value:\n            raise HRMEnvironmentError("HRM_REPO_DIR is required for HRM execution")\n        if not checkpoint_value:\n            raise HRMEnvironmentError("HRM_CHECKPOINT_PATH is required for HRM execution")\n        return cls(repo_dir=Path(repo_value), checkpoint_path=Path(checkpoint_value))\n\n    def validate(self, *, require_cuda: bool = True) -> None:\n        if not self.repo_dir.exists():\n            raise HRMEnvironmentError(f"HRM_REPO_DIR does not exist: {self.repo_dir}")\n        if not (self.repo_dir / "evaluate.py").exists():\n            raise HRMEnvironmentError(f"HRM checkout is missing evaluate.py: {self.repo_dir}")\n        if not (self.repo_dir / "dataset" / "build_arc_dataset.py").exists():\n            raise HRMEnvironmentError(\n                f"HRM checkout is missing dataset/build_arc_dataset.py: {self.repo_dir}"\n            )\n        if not self.checkpoint_path.exists():\n            raise HRMEnvironmentError(f"HRM_CHECKPOINT_PATH does not exist: {self.checkpoint_path}")\n\n        torch = self._import_torch()\n        if require_cuda and not torch.cuda.is_available():\n            raise HRMEnvironmentError("HRM execution requires CUDA; torch.cuda.is_available() is false")\n\n    def import_modules(self) -> dict[str, Any]:\n        self._add_repo_to_path()\n        modules = {}\n        for module_name in ("pretrain", "evaluate"):\n            try:\n                modules[module_name] = importlib.import_module(module_name)\n            except Exception as exc:  # pragma: no cover - depends on external HRM deps.\n                raise HRMEnvironmentError(\n                    f"failed to import HRM module {module_name!r} from {self.repo_dir}: {exc}"\n                ) from exc\n        return modules\n\n    def load_checkpoint(self) -> Any:\n        torch = self._import_torch()\n        map_location = "cuda" if torch.cuda.is_available() else "cpu"\n        try:\n            return torch.load(\n                self.checkpoint_path,\n                map_location=map_location,\n                weights_only=False,\n            )\n        except TypeError:\n            try:\n                return torch.load(self.checkpoint_path, map_location=map_location)\n            except Exception as exc:  # pragma: no cover - depends on checkpoint format.\n                raise HRMEnvironmentError(\n                    f"failed to load HRM checkpoint {self.checkpoint_path}: {exc}"\n                ) from exc\n        except Exception as exc:  # pragma: no cover - depends on checkpoint format.\n            raise HRMEnvironmentError(f"failed to load HRM checkpoint {self.checkpoint_path}: {exc}") from exc\n\n    def _add_repo_to_path(self) -> None:\n        repo = str(self.repo_dir.resolve())\n        if repo not in sys.path:\n            sys.path.insert(0, repo)\n\n    @staticmethod\n    def _import_torch() -> Any:\n        try:\n            return importlib.import_module("torch")\n        except Exception as exc:  # pragma: no cover - torch is optional locally.\n            raise HRMEnvironmentError("PyTorch is required for HRM execution") from exc\n\n\nclass HRMSolver:\n    """External HRM solver for CUDA/Kaggle smoke inference."""\n\n    def __init__(self, env: HRMEnvironment | None = None) -> None:\n        self.env = env\n\n    def solve(self, task: ArcTask) -> Prediction:\n        env = self.env or HRMEnvironment.from_env()\n        env.validate(require_cuda=True)\n        runner = HRMInferenceRunner(env)\n        return runner.solve_task(task)\n\n\n@dataclass(frozen=True)\nclass HRMInferenceRunner:\n    env: HRMEnvironment\n    num_aug: int = 0\n\n    def solve_task(self, task: ArcTask) -> Prediction:\n        run_dir = default_run_dir() / "hrm_inference" / task.id\n        raw_dir = prepare_hrm_raw_dataset(\n            (task,),\n            run_dir / "raw" / "ARC-AGI-2" / "data",\n            allow_dummy_test_outputs=True,\n        )\n        dataset_dir = run_dir / "data" / "arc-2-one-task"\n        build_hrm_dataset(\n            hrm_repo_dir=self.env.repo_dir,\n            raw_data_dir=raw_dir,\n            output_dir=dataset_dir,\n            num_aug=self.num_aug,\n        )\n        prediction_tokens = self._run_external_evaluate(dataset_dir, run_dir / "outputs")\n        return self._tokens_to_prediction(task, prediction_tokens)\n\n    def solve_tasks(self, tasks: list[ArcTask] | tuple[ArcTask, ...]) -> list[Prediction]:\n        task_list = list(tasks)\n        if not task_list:\n            return []\n        run_dir = default_run_dir() / "hrm_inference_batch"\n        raw_dir = prepare_hrm_raw_dataset(\n            task_list,\n            run_dir / "raw" / "ARC-AGI-2" / "data",\n            allow_dummy_test_outputs=True,\n        )\n        dataset_dir = run_dir / "data" / "arc-2-batch"\n        build_hrm_dataset(\n            hrm_repo_dir=self.env.repo_dir,\n            raw_data_dir=raw_dir,\n            output_dir=dataset_dir,\n            num_aug=self.num_aug,\n        )\n        prediction_tokens = self._run_external_evaluate(dataset_dir, run_dir / "outputs")\n        predictions: list[Prediction] = []\n        cursor = 0\n        for task in task_list:\n            count = len(task.test)\n            predictions.append(self._tokens_to_prediction(task, prediction_tokens[cursor : cursor + count]))\n            cursor += count\n        if cursor > len(prediction_tokens):\n            raise HRMEnvironmentError(\n                f"HRM returned {len(prediction_tokens)} predictions for {cursor} requested test inputs"\n            )\n        return predictions\n\n    def _tokens_to_prediction(\n        self,\n        task: ArcTask,\n        prediction_tokens: list[tuple[list[int], list[int]]],\n    ) -> Prediction:\n        if len(prediction_tokens) < len(task.test):\n            raise HRMEnvironmentError(\n                f"{task.id}: HRM returned {len(prediction_tokens)} predictions for "\n                f"{len(task.test)} test inputs"\n            )\n\n        attempts = []\n        for index, example in enumerate(task.test):\n            shape_hint = output_shape_hint(task, example.input)\n            top1, top2 = prediction_tokens[index]\n            attempts.append(\n                (\n                    hrm_sequence_to_grid(top1, shape_hint=shape_hint),\n                    hrm_sequence_to_grid(top2, shape_hint=shape_hint),\n                )\n            )\n        return make_prediction(task, attempts)\n\n    def _run_external_evaluate(self, dataset_dir: Path, output_dir: Path) -> list[tuple[list[int], list[int]]]:\n        torch = HRMEnvironment._import_torch()\n        modules = self.env.import_modules()\n        pretrain = modules["pretrain"]\n\n        output_dir.mkdir(parents=True, exist_ok=True)\n        config = _load_hrm_config(self.env.checkpoint_path, dataset_dir=dataset_dir, output_dir=output_dir)\n        train_loader, train_metadata = pretrain.create_dataloader(\n            config,\n            "train",\n            test_set_mode=False,\n            epochs_per_iter=1,\n            global_batch_size=config.global_batch_size,\n            rank=0,\n            world_size=1,\n        )\n        eval_loader, eval_metadata = pretrain.create_dataloader(\n            config,\n            "test",\n            test_set_mode=True,\n            epochs_per_iter=1,\n            global_batch_size=config.global_batch_size,\n            rank=0,\n            world_size=1,\n        )\n        del train_loader\n\n        train_state = pretrain.init_train_state(config, train_metadata, world_size=1)\n        checkpoint = torch.load(self.env.checkpoint_path, map_location="cuda", weights_only=False)\n        _load_hrm_checkpoint_best_effort(train_state.model, checkpoint)\n        train_state.step = 0\n        train_state.model.eval()\n        pretrain.evaluate(config, train_state, eval_loader, eval_metadata, rank=0, world_size=1)\n        return _load_decoded_hrm_predictions(output_dir)\n\n\n@dataclass(frozen=True)\nclass TTTConfig:\n    rank: int = 16\n    steps: int = 20\n    lr: float = 1e-3\n    grad_clip_norm: float = 1.0\n    # A step whose (pre-clip) LoRA gradient norm exceeds this is treated as an\n    # explosion: skipped and rolled back rather than clipped-and-applied.\n    explosion_grad_norm: float = 20.0\n    # Must be fixed and consistent across every task\'s forward passes: the\n    # puzzle embedding\'s sparse-update buffer (local_weights) is allocated\n    # once at model-init time sized to whatever global_batch_size was used\n    # then, and cannot accept a different batch size later -- confirmed by a\n    # real run: "expand: attempting to expand a dimension of length 4 -> 32"\n    # once the sizing config (32) and a per-task batch (4) diverged. 2 is the\n    # ARC-guaranteed minimum train-pair count for any task, so it\'s never\n    # dropped as an incomplete batch regardless of augmentation settings.\n    batch_size: int = 2\n    # Weight for the Genie-style background-consistency auxiliary loss (see\n    # mythos.losses.genie_background_consistency_loss): penalizes the model\n    # for changing background cells during TTT, the master plan\'s named fix\n    # for TTT "hallucinating" -- objects vanishing, backgrounds recoloring --\n    # under pure demo-pair supervision. 0 disables it.\n    genie_weight: float = 0.1\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "rank": self.rank,\n            "steps": self.steps,\n            "lr": self.lr,\n            "grad_clip_norm": self.grad_clip_norm,\n            "explosion_grad_norm": self.explosion_grad_norm,\n            "batch_size": self.batch_size,\n            "genie_weight": self.genie_weight,\n        }\n\n\nclass HRMTTTRunner:\n    """Per-task test-time training on top of the loaded HRM checkpoint.\n\n    Loads the model once and injects LoRA adapters once (the backbone is\n    frozen at injection time and never receives gradients). For each task:\n    resets the LoRA adapters to their initial no-op state, runs `ttt.steps`\n    gradient-descent steps against only that task\'s own train pairs, then\n    runs inference with the now-adapted model before moving to the next task.\n\n    The model\'s puzzle-identifier embedding table is sized once from a\n    dataset build over *all* tasks (matching the batched eval-only path) so\n    it\'s safely oversized for any single task\'s tiny per-task dataset build,\n    whose own identifier indices will always fit inside it.\n    """\n\n    def __init__(self, env: HRMEnvironment, ttt: TTTConfig | None = None, num_aug: int = 0) -> None:\n        self.env = env\n        self.ttt = ttt or TTTConfig()\n        self.num_aug = num_aug\n        self._pretrain: Any = None\n        self._train_state: Any = None\n        self._lora_report: Any = None\n        self._lora_init_snapshot: dict[str, Any] = {}\n        self._backbone_verified = False\n\n    def solve_tasks(self, tasks: list[ArcTask] | tuple[ArcTask, ...]) -> list[Prediction]:\n        task_list = list(tasks)\n        if not task_list:\n            return []\n\n        self._ensure_model_loaded(task_list)\n\n        predictions: list[Prediction] = []\n        for task in task_list:\n            predictions.append(self._solve_one_task_with_ttt(task))\n        return predictions\n\n    def _ensure_model_loaded(self, task_list: list[ArcTask]) -> None:\n        if self._train_state is not None:\n            return\n        torch = HRMEnvironment._import_torch()\n        modules = self.env.import_modules()\n        self._pretrain = modules["pretrain"]\n\n        # Dataset build over every task purely to size the model\'s architecture\n        # (vocab size, puzzle-identifier count) the same way the working\n        # eval-only batched path already does -- not used for training or eval.\n        sizing_run_dir = default_run_dir() / "hrm_ttt_sizing"\n        raw_dir = prepare_hrm_raw_dataset(\n            task_list, sizing_run_dir / "raw" / "ARC-AGI-2" / "data", allow_dummy_test_outputs=True\n        )\n        sizing_dataset_dir = sizing_run_dir / "data" / "arc-2-sizing"\n        build_hrm_dataset(\n            hrm_repo_dir=self.env.repo_dir, raw_data_dir=raw_dir, output_dir=sizing_dataset_dir, num_aug=self.num_aug\n        )\n        sizing_config = _load_hrm_config(\n            self.env.checkpoint_path, dataset_dir=sizing_dataset_dir, output_dir=sizing_run_dir / "outputs"\n        )\n        # Must match what every per-task TTT call below uses (self.ttt.batch_size),\n        # not the eval-only path\'s larger default -- see TTTConfig.batch_size.\n        sizing_config.global_batch_size = self.ttt.batch_size\n        sizing_loader, sizing_metadata = self._pretrain.create_dataloader(\n            sizing_config, "train", test_set_mode=False, epochs_per_iter=1,\n            global_batch_size=sizing_config.global_batch_size, rank=0, world_size=1,\n        )\n        del sizing_loader\n\n        train_state = self._pretrain.init_train_state(sizing_config, sizing_metadata, world_size=1)\n        checkpoint = torch.load(self.env.checkpoint_path, map_location="cuda", weights_only=False)\n        _load_hrm_checkpoint_best_effort(train_state.model, checkpoint)\n        train_state.step = 0\n\n        report = inject_lora_adapters(\n            train_state.model,\n            rank=self.ttt.rank,\n            # Attention-only adapters cap how much the model\'s actual per-task\n            # behavior can change; the MLP layers (gate_up_proj/down_proj) are\n            # roughly half the model\'s parameters and are where most of the\n            # per-token transformation logic lives. Widening the LoRA target\n            # set gives more real capacity to adapt, instead of just pushing\n            # rank/LR higher on a narrower slice of the model (confirmed\n            # unstable: v36\'s rank=64 attention-only run diverged).\n            target_patterns=("self_attn", "attn", "qkv_proj", "o_proj", "gate_up_proj", "down_proj"),\n            freeze_backbone=True,\n        )\n        print(\n            f"TTT: injected LoRA (rank={self.ttt.rank}) into {len(report.injected_modules)} module(s); "\n            f"{report.trainable_parameters} trainable / {report.frozen_parameters} frozen parameters"\n        )\n        self._lora_report = report\n        self._lora_init_snapshot = {\n            name: parameter.detach().clone()\n            for name, parameter in train_state.model.named_parameters()\n            if "lora_" in name\n        }\n        self._train_state = train_state\n\n    def _reset_lora(self) -> None:\n        torch = HRMEnvironment._import_torch()\n        with torch.no_grad():\n            for name, parameter in self._train_state.model.named_parameters():\n                snapshot = self._lora_init_snapshot.get(name)\n                if snapshot is not None:\n                    parameter.copy_(snapshot)\n\n    def _solve_one_task_with_ttt(self, task: ArcTask) -> Prediction:\n        from mythos.lora import changed_frozen_parameters, snapshot_frozen_parameters\n\n        torch = HRMEnvironment._import_torch()\n        pretrain = self._pretrain\n        train_state = self._train_state\n\n        run_dir = default_run_dir() / "hrm_ttt" / task.id\n        raw_dir = prepare_hrm_raw_dataset(\n            (task,), run_dir / "raw" / "ARC-AGI-2" / "data", allow_dummy_test_outputs=True\n        )\n        dataset_dir = run_dir / "data" / "arc-2-ttt"\n        build_hrm_dataset(\n            hrm_repo_dir=self.env.repo_dir, raw_data_dir=raw_dir, output_dir=dataset_dir, num_aug=self.num_aug\n        )\n        output_dir = run_dir / "outputs"\n        output_dir.mkdir(parents=True, exist_ok=True)\n        config = _load_hrm_config(self.env.checkpoint_path, dataset_dir=dataset_dir, output_dir=output_dir)\n        # Must equal the sizing config\'s batch size (see TTTConfig.batch_size):\n        # the puzzle embedding\'s sparse-update buffer is allocated once at\n        # model-init time and cannot accept a different batch size per task.\n        original_batch_size = config.global_batch_size\n        config.global_batch_size = self.ttt.batch_size\n        print(f"TTT: {task.id}: global_batch_size {original_batch_size} -> {config.global_batch_size}")\n\n        train_loader, _train_metadata = pretrain.create_dataloader(\n            config, "train", test_set_mode=False, epochs_per_iter=1,\n            global_batch_size=config.global_batch_size, rank=0, world_size=1,\n        )\n        eval_loader, eval_metadata = pretrain.create_dataloader(\n            config, "test", test_set_mode=True, epochs_per_iter=1,\n            global_batch_size=config.global_batch_size, rank=0, world_size=1,\n        )\n\n        self._reset_lora()\n        lora_params = lora_parameters(train_state.model)\n        optimizer = torch.optim.AdamW(lora_params, lr=self.ttt.lr)\n\n        backbone_snapshot = None\n        if not self._backbone_verified:\n            backbone_snapshot = snapshot_frozen_parameters(train_state.model)\n\n        # Diagnostic: lora_b is zero-initialized (a fresh LoRA adapter is a\n        # mathematical no-op), so any nonzero value after training proves\n        # gradients actually flowed and the optimizer actually updated\n        # something, independent of whether the eval predictions changed.\n        lora_b_before = sum(p.detach().abs().sum().item() for name, p in train_state.model.named_parameters() if name.endswith("lora_b"))\n\n        train_state.model.train()\n        carry = None\n        skipped_steps = 0\n        first_loss = None\n        last_loss = None\n        step_index = 0\n        genie_enabled = self.ttt.genie_weight > 0\n        genie_applied = 0\n        return_keys = ["logits"] if genie_enabled else []\n        for _ in range(self.ttt.steps):\n            for _set_name, batch, global_batch_size in train_loader:\n                batch = {key: (value.to("cuda") if hasattr(value, "to") else value) for key, value in batch.items()}\n                if carry is None:\n                    # initial_carry() creates some internal state tensors (e.g. the\n                    # `halted` flag) without an explicit device argument, relying on\n                    # the ambient default-device context to land them on CUDA --\n                    # confirmed both by HRM\'s own evaluate() doing the same thing\n                    # and by a real run failing with "Unhandled FakeTensor Device\n                    # Propagation for aten.where.self, found two different devices\n                    # cpu, cuda:0" without this wrapper.\n                    with torch.device("cuda"):\n                        carry = train_state.model.initial_carry(batch)\n                carry, primary_loss, _metrics, preds, _all_finish = train_state.model(\n                    carry=carry, batch=batch, return_keys=return_keys\n                )\n                loss = primary_loss\n                # Only "logits" needs to come from the model\'s return_keys -- the\n                # input grid is already in hand as batch["inputs"] (what we\'re\n                # feeding in, not something the model needs to report back).\n                # Requiring all_finish (the model\'s own halt state) first made\n                # this never fire in a real run: 0/50 steps across every task,\n                # since a single training forward call rarely reaches full ACT\n                # convergence within the step budget. logits are populated on\n                # every call regardless of halt state, so use those directly --\n                # a consistency signal on the model\'s current best guess is\n                # still useful even before it\'s fully converged.\n                if genie_enabled:\n                    try:\n                        genie_term = _batch_genie_loss(preds, batch.get("inputs"))\n                        if genie_term is not None:\n                            loss = primary_loss + self.ttt.genie_weight * genie_term\n                            genie_applied += 1\n                    except Exception as exc:  # noqa: BLE001 - auxiliary loss must never abort real TTT\n                        print(f"TTT: {task.id}: disabling Genie loss after a failure: {exc!r}")\n                        genie_enabled = False\n                        return_keys = []\n                loss_value = float(loss.detach())\n                if first_loss is None:\n                    first_loss = loss_value\n                last_loss = loss_value\n                optimizer.zero_grad(set_to_none=True)\n                (loss / max(1, global_batch_size)).backward()\n                grad_norm = torch.nn.utils.clip_grad_norm_(lora_params, self.ttt.explosion_grad_norm)\n                if not torch.isfinite(grad_norm) or grad_norm >= self.ttt.explosion_grad_norm:\n                    skipped_steps += 1\n                    optimizer.zero_grad(set_to_none=True)\n                    continue\n                torch.nn.utils.clip_grad_norm_(lora_params, self.ttt.grad_clip_norm)\n                optimizer.step()\n                step_index += 1\n        if skipped_steps:\n            print(f"TTT: {task.id}: rolled back {skipped_steps}/{self.ttt.steps} step(s) on gradient explosion")\n        if self.ttt.genie_weight > 0:\n            print(f"TTT: {task.id}: Genie consistency loss applied on {genie_applied}/{step_index} step(s)")\n\n        lora_b_after = sum(p.detach().abs().sum().item() for name, p in train_state.model.named_parameters() if name.endswith("lora_b"))\n        print(\n            f"TTT: {task.id}: loss {first_loss!r} -> {last_loss!r} over {step_index} applied step(s); "\n            f"sum(|lora_b|) {lora_b_before:.6f} -> {lora_b_after:.6f}"\n        )\n\n        if backbone_snapshot is not None:\n            changed = changed_frozen_parameters(train_state.model, backbone_snapshot)\n            if changed:\n                print(f"TTT WARNING: {len(changed)} frozen backbone parameter(s) changed during TTT: {changed[:5]}")\n            else:\n                print("TTT: verified frozen backbone parameters are unchanged after a TTT run")\n            self._backbone_verified = True\n\n        train_state.model.eval()\n        pretrain.evaluate(config, train_state, eval_loader, eval_metadata, rank=0, world_size=1)\n        tokens = _load_decoded_hrm_predictions(output_dir)\n        return HRMInferenceRunner(self.env)._tokens_to_prediction(task, tokens)\n\n\ndef _load_hrm_config(checkpoint_path: Path, *, dataset_dir: Path, output_dir: Path):  # type: ignore[no-untyped-def]\n    try:\n        import yaml\n        from pretrain import PretrainConfig\n    except Exception as exc:  # pragma: no cover - depends on external HRM deps.\n        raise HRMEnvironmentError(f"failed to import HRM config dependencies: {exc}") from exc\n\n    config_path = checkpoint_path.parent / "all_config.yaml"\n    if not config_path.exists():\n        raise HRMEnvironmentError(f"HRM checkpoint directory is missing all_config.yaml: {config_path}")\n    with config_path.open("r", encoding="utf-8") as handle:\n        config = PretrainConfig(**yaml.safe_load(handle))\n    config.data_path = str(dataset_dir)\n    config.checkpoint_path = str(output_dir)\n    config.eval_save_outputs = ["inputs", "puzzle_identifiers", "logits"]\n    if "HRM_GLOBAL_BATCH_SIZE" in os.environ:\n        config.global_batch_size = int(os.environ["HRM_GLOBAL_BATCH_SIZE"])\n    return config\n\n\ndef _batch_genie_loss(preds: Any, inputs_batch: Any) -> Any:\n    """Average Genie background-consistency loss across a training batch.\n\n    Decodes each example\'s own input tokens (from the batch fed to the model,\n    not something the model needs to report back) to a grid -- no need to\n    match against the original un-augmented ArcTask, since the preserve mask\n    is derived from the decoded grid\'s own dominant color -- and penalizes\n    the model\'s predicted logits for changing cells that should stay\n    background. Returns None (rather than raising) when preds doesn\'t\n    contain what\'s needed, so the caller\'s own try/except only has to guard\n    against genuine failures.\n    """\n    if not preds or "logits" not in preds or inputs_batch is None:\n        return None\n    logits_batch = preds["logits"]\n    if logits_batch is None or logits_batch.shape[0] == 0:\n        return None\n    # logits comes back as [batch, 900, vocab_size] -- a flat HRM token\n    # sequence, not a [H, W, 10] grid (confirmed by a real run: "logits must\n    # have rank 3 or rank 4"). Reshape to the 30x30 canvas, then slice out\n    # just the 10 color-token channels (HRM\'s vocab is PAD=0, EOS=1,\n    # colors=2..11 -- see grid_to_hrm_sequence/hrm_sequence_to_grid, the same\n    # scheme already used to decode this model\'s own predicted output\n    # tokens) since genie_background_consistency_loss compares against plain\n    # 0-9 color indices.\n    per_example_losses = []\n    for example_index in range(logits_batch.shape[0]):\n        input_tokens = inputs_batch[example_index].detach().cpu().tolist()\n        decoded_input = hrm_sequence_to_grid(input_tokens)\n        example_logits = logits_batch[example_index].reshape(ARC_MAX_SIZE, ARC_MAX_SIZE, -1)[:, :, 2:12]\n        per_example_losses.append(genie_background_consistency_loss(example_logits, decoded_input))\n    if not per_example_losses:\n        return None\n    return sum(per_example_losses) / len(per_example_losses)\n\n\ndef _load_hrm_checkpoint_best_effort(model: Any, checkpoint: dict[str, Any]) -> None:\n    """Load the pretrained checkpoint, keeping randomly-initialized weights for any\n    key whose shape doesn\'t match.\n\n    HRM\'s `puzzle_emb` is a per-puzzle lookup table sized to the exact puzzle\n    identifier vocabulary of whatever dataset it was trained on (the public\n    checkpoint: 1,045,829 entries). A dataset built from a different task set\n    (ours: 240 tasks) gets fresh, unrelated identifier indices, so this table\n    can never meaningfully transfer -- there is no "fix" for that mismatch,\n    only whether to keep evaluating with it randomly initialized (this) or\n    fail outright. Every other weight (attention/MLP layers, token/H/L init,\n    LM head) is the real pretrained model and does transfer correctly.\n    """\n    model_state = model.state_dict()\n    filtered: dict[str, Any] = {}\n    skipped: list[str] = []\n    for key, value in checkpoint.items():\n        candidates = (key, f"_orig_mod.{key}", key.removeprefix("_orig_mod."))\n        matched_key = next((name for name in candidates if name in model_state), None)\n        if matched_key is None:\n            skipped.append(f"{key}: not present in model")\n            continue\n        if tuple(model_state[matched_key].shape) != tuple(value.shape):\n            skipped.append(\n                f"{matched_key}: checkpoint={tuple(value.shape)} model={tuple(model_state[matched_key].shape)}"\n            )\n            continue\n        filtered[matched_key] = value\n\n    missing, unexpected = model.load_state_dict(filtered, strict=False, assign=True)\n    print(f"HRM checkpoint: loaded {len(filtered)}/{len(model_state)} weight tensors from the pretrained checkpoint")\n    if skipped:\n        print(f"HRM checkpoint: kept randomly-initialized (shape/name mismatch) for {len(skipped)} key(s): {skipped}")\n    if missing:\n        print(f"HRM checkpoint: {len(missing)} model key(s) had no checkpoint match: {list(missing)}")\n    if unexpected:\n        print(f"HRM checkpoint: {len(unexpected)} checkpoint key(s) were unused: {list(unexpected)}")\n\n\ndef _load_decoded_hrm_predictions(output_dir: Path) -> list[tuple[list[int], list[int]]]:\n    torch = HRMEnvironment._import_torch()\n    pred_files = sorted(output_dir.glob("step_*_all_preds.*"))\n    if not pred_files:\n        raise HRMEnvironmentError(f"HRM evaluation wrote no prediction files in {output_dir}")\n\n    raw = torch.load(pred_files[0], map_location="cpu", weights_only=False)\n    if "logits" not in raw:\n        raise HRMEnvironmentError(\n            f"HRM prediction file {pred_files[0]} is missing logits; keys={sorted(raw)}"\n        )\n    logits = raw["logits"]\n    topk = logits.topk(k=2, dim=-1).indices\n    decoded: list[tuple[list[int], list[int]]] = []\n    for row in range(topk.shape[0]):\n        top1 = topk[row, :, 0].to(dtype=torch.int64).tolist()\n        top2 = topk[row, :, 1].to(dtype=torch.int64).tolist()\n        decoded.append((top1, top2))\n    return decoded\n', 'src/mythos/solvers/pipeline.py': '"""Solver wrapper for the plan-aligned Project Mythos pipeline."""\n\nfrom __future__ import annotations\n\nfrom mythos.arc import ArcTask\nfrom mythos.models import ModelRegistry\nfrom mythos.pipeline import PipelineTrace, PlannedPipeline\nfrom mythos.submission import Prediction\n\n\nclass PlannedPipelineSolver:\n    """Solver that runs every task through the master-plan stage boundaries."""\n\n    def __init__(\n        self,\n        pipeline: PlannedPipeline | None = None,\n        *,\n        model_registry: ModelRegistry | None = None,\n        strict_models: bool = False,\n    ) -> None:\n        self.pipeline = pipeline or PlannedPipeline(\n            model_registry=model_registry,\n            strict_models=strict_models,\n        )\n        self.traces: dict[str, PipelineTrace] = {}\n        self.last_trace: PipelineTrace | None = None\n\n    def solve(self, task: ArcTask) -> Prediction:\n        result = self.pipeline.run(task)\n        self.traces[task.id] = result.trace\n        self.last_trace = result.trace\n        return result.prediction\n', 'src/mythos/submission.py': '"""Prediction and submission JSON helpers."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport json\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Mapping, Tuple\n\nfrom mythos.arc import ArcValidationError, Grid, validate_grid\n\n\n@dataclass(frozen=True)\nclass TestPrediction:\n    __test__ = False\n\n    attempt_1: Grid\n    attempt_2: Grid\n\n\n@dataclass(frozen=True)\nclass Prediction:\n    task_id: str\n    outputs: Tuple[TestPrediction, ...]\n\n\nSubmissionMap = Dict[str, Tuple[TestPrediction, ...]]\n\n\ndef prediction_to_json(prediction: Prediction) -> List[dict[str, Grid]]:\n    return [\n        {"attempt_1": item.attempt_1, "attempt_2": item.attempt_2}\n        for item in prediction.outputs\n    ]\n\n\ndef predictions_to_submission(predictions: Iterable[Prediction]) -> dict[str, List[dict[str, Grid]]]:\n    submission: dict[str, List[dict[str, Grid]]] = {}\n    for prediction in predictions:\n        if prediction.task_id in submission:\n            raise ArcValidationError(f"duplicate prediction for task {prediction.task_id}")\n        submission[prediction.task_id] = prediction_to_json(prediction)\n    if not submission:\n        raise ArcValidationError("submission must contain at least one prediction")\n    return submission\n\n\ndef validate_submission_data(data: Any) -> SubmissionMap:\n    if not isinstance(data, Mapping) or not data:\n        raise ArcValidationError("submission must be a non-empty object keyed by task id")\n    validated: SubmissionMap = {}\n    for task_id, raw_outputs in data.items():\n        if not isinstance(raw_outputs, list) or not raw_outputs:\n            raise ArcValidationError(f"{task_id} must contain a non-empty list of test outputs")\n        outputs: List[TestPrediction] = []\n        for index, raw_output in enumerate(raw_outputs):\n            if not isinstance(raw_output, Mapping):\n                raise ArcValidationError(f"{task_id}[{index}] must be an object")\n            if "attempt_1" not in raw_output or "attempt_2" not in raw_output:\n                raise ArcValidationError(f"{task_id}[{index}] must contain attempt_1 and attempt_2")\n            outputs.append(\n                TestPrediction(\n                    attempt_1=validate_grid(raw_output["attempt_1"], field=f"{task_id}[{index}].attempt_1"),\n                    attempt_2=validate_grid(raw_output["attempt_2"], field=f"{task_id}[{index}].attempt_2"),\n                )\n            )\n        validated[str(task_id)] = tuple(outputs)\n    return validated\n\n\ndef write_submission(predictions: Iterable[Prediction], path: str | Path) -> None:\n    output_path = Path(path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    data = predictions_to_submission(predictions)\n    with output_path.open("w", encoding="utf-8") as handle:\n        json.dump(data, handle, indent=2)\n        handle.write("\\n")\n\n\ndef load_submission(path: str | Path) -> SubmissionMap:\n    input_path = Path(path)\n    try:\n        with input_path.open("r", encoding="utf-8") as handle:\n            raw = json.load(handle)\n    except json.JSONDecodeError as exc:\n        raise ArcValidationError(f"{input_path} is not valid JSON: {exc}") from exc\n    return validate_submission_data(raw)\n', 'src/mythos/text_reasoning.py': '"""Optional HRM-Text rule generation for ARC tasks."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Sequence\n\nfrom mythos.arc import ArcTask, Grid\nfrom mythos.features import DEFAULT_RULE_DIM\n\n\nclass HRMTextError(RuntimeError):\n    """Raised when optional HRM-Text inference cannot run."""\n\n\n@dataclass(frozen=True)\nclass HRMTextRuleResult:\n    description: str\n    vector: tuple[float, ...]\n    model_root: str\n\n\ndef generate_hrm_text_rule(\n    task: ArcTask,\n    model_path: str | Path,\n    *,\n    max_new_tokens: int = 96,\n    device: str | None = None,\n    vector_dim: int = DEFAULT_RULE_DIM,\n) -> HRMTextRuleResult:\n    """Run a Hugging Face text model to produce a rule description/vector."""\n\n    try:\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n    except Exception as exc:  # pragma: no cover - optional dependency.\n        raise HRMTextError("transformers and torch are required for HRM-Text inference") from exc\n\n    root = _model_root(model_path)\n    selected_device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    try:\n        tokenizer = AutoTokenizer.from_pretrained(root, trust_remote_code=True)\n        model = AutoModelForCausalLM.from_pretrained(\n            root,\n            trust_remote_code=True,\n            torch_dtype=torch.float16 if selected_device == "cuda" else torch.float32,\n        ).to(selected_device)\n    except Exception as exc:  # pragma: no cover - depends on external model files.\n        raise HRMTextError(f"failed to load HRM-Text model from {root}: {exc}") from exc\n\n    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    prompt = format_arc_rule_prompt(task)\n    encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(selected_device)\n    try:\n        with torch.no_grad():\n            generated = model.generate(\n                **encoded,\n                do_sample=False,\n                max_new_tokens=max_new_tokens,\n                pad_token_id=tokenizer.pad_token_id,\n            )\n            forward = model(\n                generated,\n                output_hidden_states=True,\n                use_cache=False,\n            )\n    except Exception as exc:  # pragma: no cover - depends on external model behavior.\n        raise HRMTextError(f"HRM-Text forward/generation failed: {exc}") from exc\n\n    generated_text = tokenizer.decode(generated[0], skip_special_tokens=True)\n    description = _extract_answer(prompt, generated_text)\n    vector = _hidden_state_to_vector(forward.hidden_states[-1][0], vector_dim)\n    return HRMTextRuleResult(description=description, vector=vector, model_root=str(root))\n\n\ndef format_arc_rule_prompt(task: ArcTask) -> str:\n    examples = []\n    for index, example in enumerate(task.train):\n        examples.append(\n            f"Train {index} input:\\n{_grid_text(example.input)}\\n"\n            f"Train {index} output:\\n{_grid_text(example.output or example.input)}"\n        )\n    tests = []\n    for index, example in enumerate(task.test):\n        tests.append(f"Test {index} input:\\n{_grid_text(example.input)}")\n    return (\n        "You are solving an ARC abstract reasoning task. Infer the rule from "\n        "the train pairs and describe the transformation in one concise sentence.\\n\\n"\n        + "\\n\\n".join(examples)\n        + "\\n\\n"\n        + "\\n\\n".join(tests)\n        + "\\n\\nRule:"\n    )\n\n\ndef _model_root(path: str | Path) -> Path:\n    candidate = Path(path)\n    return candidate.parent if candidate.is_file() else candidate\n\n\ndef _grid_text(grid: Grid) -> str:\n    return "\\n".join(" ".join(str(cell) for cell in row) for row in grid)\n\n\ndef _extract_answer(prompt: str, generated_text: str) -> str:\n    if generated_text.startswith(prompt):\n        generated_text = generated_text[len(prompt) :]\n    answer = generated_text.strip()\n    return answer or "No HRM-Text rule text generated."\n\n\ndef _hidden_state_to_vector(hidden_state, dim: int) -> tuple[float, ...]:  # type: ignore[no-untyped-def]\n    if dim <= 0:\n        raise ValueError("vector_dim must be positive")\n    pooled = hidden_state.float().mean(dim=0).detach().cpu()\n    if pooled.numel() < dim:\n        values = pooled.tolist() + [0.0 for _ in range(dim - pooled.numel())]\n    else:\n        chunk_size = max(1, pooled.numel() // dim)\n        values = []\n        for index in range(dim):\n            start = index * chunk_size\n            end = pooled.numel() if index == dim - 1 else min(pooled.numel(), start + chunk_size)\n            values.append(float(pooled[start:end].mean()))\n    return _normalize(values[:dim])\n\n\ndef _normalize(values: Sequence[float]) -> tuple[float, ...]:\n    norm = sum(value * value for value in values) ** 0.5\n    if norm == 0.0:\n        return tuple(round(value, 6) for value in values)\n    return tuple(round(value / norm, 6) for value in values)\n', 'src/mythos/train_projection.py': '"""CLI for training the JEPA-to-HRM projection checkpoint."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\n\nfrom mythos.arc import ArcValidationError, attach_solutions, load_challenges, load_solutions\nfrom mythos.training import JepaProjectionConfig, train_jepa_projection\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Train the Mythos I-JEPA projection checkpoint.")\n    parser.add_argument("--challenges", required=True)\n    parser.add_argument("--solutions")\n    parser.add_argument("--out", required=True)\n    parser.add_argument("--steps", type=int, default=200)\n    parser.add_argument("--lr", type=float, default=1e-3)\n    parser.add_argument("--input-dim", type=int, default=1280)\n    parser.add_argument("--output-dim", type=int, default=768)\n    parser.add_argument("--device")\n    parser.add_argument("--include-test-solutions", action="store_true")\n    args = parser.parse_args(argv)\n\n    try:\n        tasks = load_challenges(args.challenges)\n        if args.solutions:\n            tasks = attach_solutions(tasks, load_solutions(args.solutions))\n        result = train_jepa_projection(\n            tasks.values(),\n            checkpoint_path=args.out,\n            config=JepaProjectionConfig(input_dim=args.input_dim, output_dim=args.output_dim),\n            steps=args.steps,\n            lr=args.lr,\n            device=args.device,\n            include_test_solutions=args.include_test_solutions,\n        )\n    except (ArcValidationError, RuntimeError, ValueError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n\n    print(json.dumps(result.to_dict(), indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/train_world_model.py': '"""CLI for training the Mythos world-model transition checkpoint."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\n\nfrom mythos.arc import ArcValidationError, attach_solutions, load_challenges, load_solutions\nfrom mythos.training import WorldModelConfig, train_world_model\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Train the Mythos world-model checkpoint.")\n    parser.add_argument("--challenges", required=True)\n    parser.add_argument("--solutions")\n    parser.add_argument("--out", required=True)\n    parser.add_argument("--steps", type=int, default=300)\n    parser.add_argument("--lr", type=float, default=1e-3)\n    parser.add_argument("--z-dim", type=int, default=768)\n    parser.add_argument("--rule-dim", type=int, default=4)\n    parser.add_argument("--hidden-dim", type=int, default=3072)\n    parser.add_argument("--device")\n    parser.add_argument("--include-test-solutions", action="store_true")\n    args = parser.parse_args(argv)\n\n    try:\n        tasks = load_challenges(args.challenges)\n        if args.solutions:\n            tasks = attach_solutions(tasks, load_solutions(args.solutions))\n        result = train_world_model(\n            tasks.values(),\n            checkpoint_path=args.out,\n            config=WorldModelConfig(\n                z_dim=args.z_dim,\n                rule_dim=args.rule_dim,\n                hidden_dim=args.hidden_dim,\n            ),\n            steps=args.steps,\n            lr=args.lr,\n            device=args.device,\n            include_test_solutions=args.include_test_solutions,\n        )\n    except (ArcValidationError, RuntimeError, ValueError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n\n    print(json.dumps(result.to_dict(), indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/training.py': '"""Training helpers for Project Mythos planned stages."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nimport time\nfrom typing import Iterable\n\nfrom mythos.arc import ArcTask\nfrom mythos.features import (\n    DEFAULT_HRM_FEATURE_DIM,\n    DEFAULT_JEPA_FEATURE_DIM,\n    DEFAULT_RULE_DIM,\n    grid_to_feature_vector,\n    iter_all_supervised_grid_pairs,\n    iter_train_grid_pairs,\n    max_pairwise_cosine,\n    task_rule_vector,\n)\nfrom mythos.lora import (\n    changed_frozen_parameters,\n    inject_lora_adapters,\n    lora_parameters,\n    save_lora_checkpoint,\n    snapshot_frozen_parameters,\n)\n\n\ntry:  # Keep imports cheap for CLI validation paths that do not train.\n    from torch import nn as _OPTIONAL_NN\nexcept Exception:  # pragma: no cover - depends on optional torch install.\n    _OPTIONAL_NN = None\n\n\ndef _torch():\n    try:\n        import torch\n    except Exception as exc:  # pragma: no cover - depends on optional torch install.\n        raise RuntimeError("PyTorch is required for Mythos training helpers") from exc\n    return torch\n\n\ndef _nn():\n    try:\n        from torch import nn\n    except Exception as exc:  # pragma: no cover - depends on optional torch install.\n        raise RuntimeError("PyTorch is required for Mythos training helpers") from exc\n    return nn\n\n\n@dataclass(frozen=True)\nclass JepaProjectionConfig:\n    input_dim: int = DEFAULT_JEPA_FEATURE_DIM\n    output_dim: int = DEFAULT_HRM_FEATURE_DIM\n    seed: int = 7\n\n\n@dataclass(frozen=True)\nclass WorldModelConfig:\n    z_dim: int = DEFAULT_HRM_FEATURE_DIM\n    rule_dim: int = DEFAULT_RULE_DIM\n    hidden_dim: int = 3072\n    seed: int = 11\n\n\n@dataclass(frozen=True)\nclass TrainingResult:\n    stage: str\n    checkpoint_path: str | None\n    steps: int\n    examples: int\n    initial_loss: float\n    final_loss: float\n    elapsed_seconds: float\n    extra: dict[str, object]\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "stage": self.stage,\n            "checkpoint_path": self.checkpoint_path,\n            "steps": self.steps,\n            "examples": self.examples,\n            "initial_loss": self.initial_loss,\n            "final_loss": self.final_loss,\n            "elapsed_seconds": self.elapsed_seconds,\n            "extra": self.extra,\n        }\n\n\n@dataclass(frozen=True)\nclass TTTSmokeResult:\n    steps: int\n    rank: int\n    injected_modules: tuple[str, ...]\n    initial_loss: float\n    final_loss: float\n    first_backward_seconds: float\n    frozen_parameter_changes: tuple[str, ...]\n    checkpoint_path: str | None\n\n    @property\n    def backbone_frozen(self) -> bool:\n        return not self.frozen_parameter_changes\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "steps": self.steps,\n            "rank": self.rank,\n            "injected_modules": list(self.injected_modules),\n            "initial_loss": self.initial_loss,\n            "final_loss": self.final_loss,\n            "first_backward_seconds": self.first_backward_seconds,\n            "frozen_parameter_changes": list(self.frozen_parameter_changes),\n            "backbone_frozen": self.backbone_frozen,\n            "checkpoint_path": self.checkpoint_path,\n        }\n\n\n_BASE_MODULE = _OPTIONAL_NN.Module if _OPTIONAL_NN is not None else object\n\n\nclass JepaProjection(_BASE_MODULE):\n    """Projection layer from I-JEPA/ARC features into the HRM feature width."""\n\n    def __init__(self, config: JepaProjectionConfig) -> None:\n        nn = _nn()\n        super().__init__()\n        self.config = config\n        self.norm = nn.LayerNorm(config.input_dim, elementwise_affine=False)\n        self.proj = nn.Linear(config.input_dim, config.output_dim)\n\n    def forward(self, inputs):  # type: ignore[no-untyped-def]\n        return self.proj(self.norm(inputs))\n\n\nclass WorldModelMLP(_BASE_MODULE):\n    """Two-layer transition model f(z_input, v_rule) -> z_output."""\n\n    def __init__(self, config: WorldModelConfig) -> None:\n        nn = _nn()\n        super().__init__()\n        self.config = config\n        self.net = nn.Sequential(\n            nn.Linear(config.z_dim + config.rule_dim, config.hidden_dim),\n            nn.GELU(),\n            nn.Linear(config.hidden_dim, config.z_dim),\n        )\n\n    def forward(self, z_input, rule):  # type: ignore[no-untyped-def]\n        torch = _torch()\n        return self.net(torch.cat([z_input, rule], dim=-1))\n\n\ndef train_jepa_projection(\n    tasks: Iterable[ArcTask],\n    *,\n    checkpoint_path: str | Path | None = None,\n    config: JepaProjectionConfig | None = None,\n    steps: int = 200,\n    lr: float = 1e-3,\n    device: str | None = None,\n    include_test_solutions: bool = False,\n) -> TrainingResult:\n    """Train only the ARC-to-HRM projection checkpoint."""\n\n    torch = _torch()\n    nn = _nn()\n    cfg = config or JepaProjectionConfig()\n    torch.manual_seed(cfg.seed)\n    started = time.perf_counter()\n\n    pairs = list(\n        iter_all_supervised_grid_pairs(tasks) if include_test_solutions else iter_train_grid_pairs(tasks)\n    )\n    if not pairs:\n        raise ValueError("no supervised ARC grid pairs available for projection training")\n\n    selected_device = _select_device(device)\n    x = torch.tensor(\n        [grid_to_feature_vector(pair.input, cfg.input_dim) for pair in pairs],\n        dtype=torch.float32,\n        device=selected_device,\n    )\n    y = torch.tensor(\n        [grid_to_feature_vector(pair.output, cfg.output_dim) for pair in pairs],\n        dtype=torch.float32,\n        device=selected_device,\n    )\n\n    model = JepaProjection(cfg).to(selected_device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)\n    loss_fn = nn.MSELoss()\n\n    with torch.no_grad():\n        initial_loss = float(loss_fn(model(x), y).detach().cpu())\n    for _ in range(max(0, steps)):\n        optimizer.zero_grad(set_to_none=True)\n        loss = loss_fn(model(x), y)\n        loss.backward()\n        optimizer.step()\n    with torch.no_grad():\n        final_loss = float(loss_fn(model(x), y).detach().cpu())\n\n    output = None\n    if checkpoint_path is not None:\n        output = save_projection_checkpoint(model, checkpoint_path)\n\n    diversity = max_pairwise_cosine(\n        [grid_to_feature_vector(pair.input, cfg.input_dim) for pair in pairs[: min(10, len(pairs))]]\n    )\n    return TrainingResult(\n        stage="jepa_projection",\n        checkpoint_path=str(output) if output is not None else None,\n        steps=steps,\n        examples=len(pairs),\n        initial_loss=round(initial_loss, 8),\n        final_loss=round(final_loss, 8),\n        elapsed_seconds=round(time.perf_counter() - started, 3),\n        extra={\n            "config": asdict(cfg),\n            "device": selected_device,\n            "max_pairwise_input_cosine": round(diversity, 6),\n            "trainable_parameters": sum(parameter.numel() for parameter in model.parameters()),\n        },\n    )\n\n\ndef save_projection_checkpoint(model: JepaProjection, path: str | Path) -> Path:\n    torch = _torch()\n    output_path = Path(path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(\n        {\n            "kind": "mythos_jepa_projection",\n            "config": asdict(model.config),\n            "state_dict": model.state_dict(),\n        },\n        output_path,\n    )\n    return output_path\n\n\ndef load_projection_checkpoint(path: str | Path, *, device: str | None = None) -> JepaProjection:\n    torch = _torch()\n    selected_device = _select_device(device)\n    raw = torch.load(path, map_location=selected_device, weights_only=False)\n    config = JepaProjectionConfig(**raw["config"])\n    model = JepaProjection(config).to(selected_device)\n    model.load_state_dict(raw["state_dict"])\n    model.eval()\n    return model\n\n\ndef train_world_model(\n    tasks: Iterable[ArcTask],\n    *,\n    checkpoint_path: str | Path | None = None,\n    config: WorldModelConfig | None = None,\n    steps: int = 300,\n    lr: float = 1e-3,\n    device: str | None = None,\n    include_test_solutions: bool = False,\n) -> TrainingResult:\n    """Train the two-layer transition world model from ARC before/after pairs."""\n\n    torch = _torch()\n    nn = _nn()\n    cfg = config or WorldModelConfig()\n    torch.manual_seed(cfg.seed)\n    started = time.perf_counter()\n\n    task_list = list(tasks)\n    pairs = list(\n        iter_all_supervised_grid_pairs(task_list) if include_test_solutions else iter_train_grid_pairs(task_list)\n    )\n    task_by_id = {task.id: task for task in task_list}\n    if not pairs:\n        raise ValueError("no supervised ARC grid pairs available for world-model training")\n\n    selected_device = _select_device(device)\n    z_input = torch.tensor(\n        [grid_to_feature_vector(pair.input, cfg.z_dim) for pair in pairs],\n        dtype=torch.float32,\n        device=selected_device,\n    )\n    rules = torch.tensor(\n        [task_rule_vector(task_by_id[pair.task_id], cfg.rule_dim) for pair in pairs],\n        dtype=torch.float32,\n        device=selected_device,\n    )\n    z_target = torch.tensor(\n        [grid_to_feature_vector(pair.output, cfg.z_dim) for pair in pairs],\n        dtype=torch.float32,\n        device=selected_device,\n    )\n\n    model = WorldModelMLP(cfg).to(selected_device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)\n    loss_fn = nn.MSELoss()\n\n    with torch.no_grad():\n        initial_loss = float(loss_fn(model(z_input, rules), z_target).detach().cpu())\n    for _ in range(max(0, steps)):\n        optimizer.zero_grad(set_to_none=True)\n        loss = loss_fn(model(z_input, rules), z_target)\n        loss.backward()\n        optimizer.step()\n    with torch.no_grad():\n        final_loss = float(loss_fn(model(z_input, rules), z_target).detach().cpu())\n\n    output = None\n    if checkpoint_path is not None:\n        output = save_world_model_checkpoint(model, checkpoint_path)\n\n    return TrainingResult(\n        stage="world_model",\n        checkpoint_path=str(output) if output is not None else None,\n        steps=steps,\n        examples=len(pairs),\n        initial_loss=round(initial_loss, 8),\n        final_loss=round(final_loss, 8),\n        elapsed_seconds=round(time.perf_counter() - started, 3),\n        extra={\n            "config": asdict(cfg),\n            "device": selected_device,\n            "trainable_parameters": sum(parameter.numel() for parameter in model.parameters()),\n        },\n    )\n\n\ndef save_world_model_checkpoint(model: WorldModelMLP, path: str | Path) -> Path:\n    torch = _torch()\n    output_path = Path(path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(\n        {\n            "kind": "mythos_world_model",\n            "config": asdict(model.config),\n            "state_dict": model.state_dict(),\n        },\n        output_path,\n    )\n    return output_path\n\n\ndef load_world_model_checkpoint(path: str | Path, *, device: str | None = None) -> WorldModelMLP:\n    torch = _torch()\n    selected_device = _select_device(device)\n    raw = torch.load(path, map_location=selected_device, weights_only=False)\n    config = WorldModelConfig(**raw["config"])\n    model = WorldModelMLP(config).to(selected_device)\n    model.load_state_dict(raw["state_dict"])\n    model.eval()\n    return model\n\n\ndef run_ttt_lora_smoke(\n    *,\n    rank: int = 16,\n    steps: int = 50,\n    dim: int = 32,\n    batch_size: int = 8,\n    lr: float = 1e-2,\n    device: str | None = None,\n    checkpoint_path: str | Path | None = None,\n) -> TTTSmokeResult:\n    """Run a small LoRA-only optimization to validate the TTT mechanics."""\n\n    torch = _torch()\n    nn = _nn()\n    selected_device = _select_device(device)\n    torch.manual_seed(23)\n\n    class TinyAttentionModel(nn.Module):\n        def __init__(self) -> None:\n            super().__init__()\n            self.attention_q = nn.Linear(dim, dim)\n            self.attention_out = nn.Linear(dim, dim)\n            self.mlp = nn.Sequential(nn.GELU(), nn.Linear(dim, dim))\n\n        def forward(self, x):  # type: ignore[no-untyped-def]\n            return self.mlp(self.attention_out(torch.tanh(self.attention_q(x))))\n\n    model = TinyAttentionModel().to(selected_device)\n    report = inject_lora_adapters(\n        model,\n        rank=rank,\n        target_patterns=("attention",),\n        fallback_to_all_linear=False,\n        freeze_backbone=True,\n    )\n    snapshot = snapshot_frozen_parameters(model)\n    optimizer = torch.optim.AdamW(lora_parameters(model), lr=lr)\n    loss_fn = nn.MSELoss()\n    inputs = torch.randn(batch_size, dim, device=selected_device)\n    targets = torch.flip(inputs, dims=(-1,))\n\n    with torch.no_grad():\n        initial_loss = float(loss_fn(model(inputs), targets).detach().cpu())\n\n    first_backward_seconds = 0.0\n    for step_index in range(max(0, steps)):\n        optimizer.zero_grad(set_to_none=True)\n        outputs = model(inputs)\n        loss = loss_fn(outputs, targets)\n        started = time.perf_counter()\n        loss.backward()\n        if step_index == 0:\n            first_backward_seconds = time.perf_counter() - started\n        optimizer.step()\n\n    with torch.no_grad():\n        final_loss = float(loss_fn(model(inputs), targets).detach().cpu())\n    changed = changed_frozen_parameters(model, snapshot)\n\n    output = None\n    if checkpoint_path is not None:\n        output = save_lora_checkpoint(\n            model,\n            checkpoint_path,\n            metadata={\n                "rank": rank,\n                "steps": steps,\n                "dim": dim,\n                "batch_size": batch_size,\n                "smoke": True,\n            },\n        )\n\n    return TTTSmokeResult(\n        steps=steps,\n        rank=rank,\n        injected_modules=report.injected_modules,\n        initial_loss=round(initial_loss, 8),\n        final_loss=round(final_loss, 8),\n        first_backward_seconds=round(first_backward_seconds, 6),\n        frozen_parameter_changes=changed,\n        checkpoint_path=str(output) if output is not None else None,\n    )\n\n\ndef adaptive_ttt_should_stop(\n    losses: Iterable[float],\n    *,\n    min_delta: float = 1e-4,\n    patience: int = 5,\n) -> bool:\n    """Return True when recent TTT loss improvements have flattened."""\n\n    collected = list(losses)\n    if len(collected) <= patience:\n        return False\n    recent = collected[-(patience + 1) :]\n    improvements = [recent[index] - recent[index + 1] for index in range(len(recent) - 1)]\n    return all(improvement < min_delta for improvement in improvements)\n\n\ndef _select_device(device: str | None) -> str:\n    if device:\n        return device\n    torch = _torch()\n    return "cuda" if torch.cuda.is_available() else "cpu"\n', 'src/mythos/ttt_smoke.py': '"""CLI for validating LoRA-only test-time-training mechanics."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\n\nfrom mythos.training import run_ttt_lora_smoke\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Run a Mythos LoRA/TTT smoke test.")\n    parser.add_argument("--out")\n    parser.add_argument("--rank", type=int, default=16)\n    parser.add_argument("--steps", type=int, default=50)\n    parser.add_argument("--dim", type=int, default=32)\n    parser.add_argument("--batch-size", type=int, default=8)\n    parser.add_argument("--lr", type=float, default=1e-2)\n    parser.add_argument("--device")\n    args = parser.parse_args(argv)\n\n    try:\n        result = run_ttt_lora_smoke(\n            rank=args.rank,\n            steps=args.steps,\n            dim=args.dim,\n            batch_size=args.batch_size,\n            lr=args.lr,\n            device=args.device,\n            checkpoint_path=args.out,\n        )\n    except (RuntimeError, ValueError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n\n    print(json.dumps(result.to_dict(), indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/validate.py': '"""CLI for ARC challenge validation."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\n\nfrom mythos.arc import ArcValidationError, load_challenges\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Validate an ARC challenges.json file.")\n    parser.add_argument("path", help="Path to ARC-style challenges JSON.")\n    args = parser.parse_args(argv)\n\n    try:\n        tasks = load_challenges(args.path)\n    except ArcValidationError as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n    train_count = sum(len(task.train) for task in tasks.values())\n    test_count = sum(len(task.test) for task in tasks.values())\n    print(f"OK: {len(tasks)} tasks, {train_count} train examples, {test_count} test items")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'}

EMBED_ROOT = Path(os.environ.get('MYTHOS_EMBED_ROOT', '/kaggle/working/project_mythos_embedded'))
if not EMBED_ROOT.parent.exists():
    EMBED_ROOT = Path.cwd() / 'project_mythos_embedded'

for relative_path, content in EMBEDDED_FILES.items():
    path = EMBED_ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')

SRC_DIR = EMBED_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Embedded Mythos package written to:', SRC_DIR)
print('Embedded files:', len(EMBEDDED_FILES))


## 2. Configuration

In [ ]:
from pathlib import Path
import json
import os
import time

DATA_DIR = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2')
# ARCHITECTURE pass: hyperparameter tuning alone plateaued (v29-v37: stable,
# decreasing loss but 0 exact matches regardless of steps/rank/lr). Testing two
# real architectural changes together -- Genie background-consistency loss (was
# implemented in mythos.losses but never wired into any training loop) and wider
# LoRA target modules (MLP layers, not just attention) -- against the evaluation
# split's known solutions before spending the daily submission quota again.
SPLIT = 'test'  # 'evaluation' was used for diagnostic runs against known solutions; 'test' is the real submission split
os.environ.setdefault('MYTHOS_TTT_STEPS', '200')
os.environ.setdefault('MYTHOS_TTT_NUM_AUG', '8')
os.environ.setdefault('MYTHOS_TTT_RANK', '16')
os.environ.setdefault('MYTHOS_TTT_GENIE_WEIGHT', '0.01')
os.environ.setdefault('MYTHOS_TTT_LR', '1e-4')
os.environ.setdefault('MYTHOS_TTT_BATCH_SIZE', '2')
SOLVER_NAME = 'hrm'  # 'pipeline', 'baseline', 'fixture', or 'hrm'
MODEL_MODE = 'fallback'  # 'fallback' or 'strict' (irrelevant for SOLVER_NAME='hrm', which bypasses ModelRegistry)
# Kaggle competition reruns are internet-disabled, so the HRM repo + checkpoint
# are pre-staged as a Kaggle Dataset (ankitdash24/hrm-arc2-checkpoint) instead of
# downloaded live -- verified working end-to-end first with AUTO_DOWNLOAD+internet
# in a dev run, then pinned here for the actually-submittable configuration.
AUTO_DOWNLOAD_GIT_CODE = False
AUTO_DOWNLOAD_HF_MODELS = False
AUTO_DOWNLOAD_DIRECT_CHECKPOINTS = False
AUTO_DISCOVER_MODELS = True
os.environ.setdefault('HRM_REPO_DIR', '/kaggle/input/hrm-arc2-checkpoint/hrm-repo')
os.environ.setdefault('HRM_CHECKPOINT_PATH', '/kaggle/input/hrm-arc2-checkpoint/hrm-checkpoint/checkpoint')
OUTPUT_PATH = Path('/kaggle/working/submission.json')
RUN_HRM_SMOKE = False

# Training/checkpoint-producing stages. Keep all False for final rerun unless needed.
RUN_TRAINING_STAGES = False
TRAIN_JEPA_PROJECTION = False
TRAIN_WORLD_MODEL = False
RUN_TTT_SMOKE = False
ENABLE_REAL_HRM_INFERENCE = True
ENABLE_REAL_JEPA = False
ENABLE_HRM_TEXT = False
ENABLE_TTT_IN_PIPELINE = True  # drives MYTHOS_ENABLE_TTT for SOLVER_NAME='hrm' too, not just the pipeline solver
CHECKPOINT_DIR = Path('/kaggle/working/mythos_checkpoints')
IJEPA_PROJECTION_OUTPUT = CHECKPOINT_DIR / 'ijepa_projection.pt'
WORLD_MODEL_OUTPUT = CHECKPOINT_DIR / 'world_model.pt'
TTT_LORA_OUTPUT = CHECKPOINT_DIR / 'ttt_lora_smoke.pt'
JEPA_PROJECTION_STEPS = 200
WORLD_MODEL_STEPS = 300
TTT_SMOKE_STEPS = 50

os.environ['MYTHOS_ENABLE_REAL_HRM'] = '1' if ENABLE_REAL_HRM_INFERENCE else '0'
os.environ['MYTHOS_ENABLE_REAL_JEPA'] = '1' if ENABLE_REAL_JEPA else '0'
os.environ['MYTHOS_ENABLE_HRM_TEXT'] = '1' if ENABLE_HRM_TEXT else '0'
os.environ['MYTHOS_ENABLE_TTT'] = '1' if ENABLE_TTT_IN_PIPELINE else '0'
# HRM's own eval path logs to Weights & Biases; without this it can block on an
# interactive API-key prompt in a non-interactive kernel and hang out the session.
os.environ.setdefault('WANDB_MODE', 'offline')
# HRM's public checkpoint was trained for 8-GPU distributed batches; keep eval batches
# small since this pipeline calls HRM's eval loop with world_size=1 on Kaggle's GPU(s).
os.environ.setdefault('HRM_GLOBAL_BATCH_SIZE', '32')

# Verified public defaults looked up from official sources.
os.environ.setdefault('HRM_GIT_REPO_URL', 'https://github.com/sapientinc/HRM.git')
os.environ.setdefault('HRM_HF_REPO_ID', 'sapientinc/HRM-checkpoint-ARC-2')
os.environ.setdefault('HRM_HF_CHECKPOINT_GLOB', 'checkpoint')
os.environ.setdefault('IJEPA_HF_REPO_ID', 'facebook/ijepa_vith14_1k')
os.environ.setdefault('IJEPA_HF_CHECKPOINT_GLOB', 'model.safetensors')
os.environ.setdefault('HRM_TEXT_HF_REPO_ID', 'sapientinc/HRM-Text-1B')
os.environ.setdefault('HRM_TEXT_HF_CHECKPOINT_GLOB', 'model.safetensors')

# Components intentionally left unset because no verified public checkpoint ID was found.
# Train/fine-tune these and add your own dataset/HF IDs when available:
# - IJEPA_PROJECTION_HF_REPO_ID / IJEPA_PROJECTION_CHECKPOINT_PATH
# - WORLD_MODEL_HF_REPO_ID / WORLD_MODEL_CHECKPOINT_PATH
# - TTT_LORA_HF_REPO_ID / TTT_LORA_CHECKPOINT_PATH

# For real transformers I-JEPA, stage the Hugging Face snapshot as a Kaggle Dataset
# and set IJEPA_CHECKPOINT_PATH to any file inside that snapshot, or let autodiscovery find it.

# Optional real-model inputs. Set these if you have additional public/private model repos.
# os.environ['IJEPA_HF_REPO_ID'] = '<org-or-user>/<ijepa-model-repo>'
# os.environ['IJEPA_HF_CHECKPOINT_GLOB'] = '*.pt'
# os.environ['IJEPA_PROJECTION_HF_REPO_ID'] = '<org-or-user>/<projection-repo>'
# os.environ['HRM_TEXT_HF_REPO_ID'] = '<org-or-user>/<hrm-text-model-repo>'
# os.environ['WORLD_MODEL_HF_REPO_ID'] = '<org-or-user>/<world-model-repo>'
# os.environ['TTT_LORA_HF_REPO_ID'] = '<org-or-user>/<lora-repo>'
# os.environ['HRM_HF_REPO_ID'] = '<org-or-user>/<hrm-model-repo>'
# os.environ['HRM_HF_CHECKPOINT_GLOB'] = '*.pt'

# Or set explicit Kaggle input paths when internet/download is unavailable.
# os.environ['IJEPA_CHECKPOINT_PATH'] = '/kaggle/input/<ijepa-hf-snapshot>/model.safetensors'
# os.environ['IJEPA_PROJECTION_CHECKPOINT_PATH'] = '/kaggle/input/<projection>/ijepa_projection.pt'
# os.environ['HRM_TEXT_REPO_DIR'] = '/kaggle/input/<hrm-text-code>/hrm-text'
# os.environ['HRM_TEXT_CHECKPOINT_PATH'] = '/kaggle/input/<hrm-text-checkpoint>/checkpoint.pt'
# os.environ['WORLD_MODEL_CHECKPOINT_PATH'] = '/kaggle/input/<world-model>/world_model.pt'
# os.environ['TTT_LORA_CHECKPOINT_PATH'] = '/kaggle/input/<lora>/lora.pt'
# os.environ['HRM_REPO_DIR'] = '/kaggle/input/<hrm-code>/HRM'
# os.environ['HRM_CHECKPOINT_PATH'] = '/kaggle/input/<hrm-checkpoint>/checkpoint.pt'

# HRM uses flash-attn in its attention path. For final offline reruns, pre-stage a wheel
# built against Kaggle's CUDA/PyTorch image instead of building from source at submission time.

print('DATA_DIR =', DATA_DIR)
print('SPLIT =', SPLIT)
print('SOLVER_NAME =', SOLVER_NAME)
print('MODEL_MODE =', MODEL_MODE)
print('AUTO_DOWNLOAD_GIT_CODE =', AUTO_DOWNLOAD_GIT_CODE)
print('AUTO_DOWNLOAD_HF_MODELS =', AUTO_DOWNLOAD_HF_MODELS)
print('AUTO_DOWNLOAD_DIRECT_CHECKPOINTS =', AUTO_DOWNLOAD_DIRECT_CHECKPOINTS)
print('AUTO_DISCOVER_MODELS =', AUTO_DISCOVER_MODELS)
print('OUTPUT_PATH =', OUTPUT_PATH)
print('RUN_TRAINING_STAGES =', RUN_TRAINING_STAGES)
print('ENABLE_REAL_HRM_INFERENCE =', ENABLE_REAL_HRM_INFERENCE)
print('ENABLE_REAL_JEPA =', ENABLE_REAL_JEPA)
print('ENABLE_HRM_TEXT =', ENABLE_HRM_TEXT)
print('ENABLE_TTT_IN_PIPELINE =', ENABLE_TTT_IN_PIPELINE)
print('MYTHOS_MAX_TASKS =', os.environ.get('MYTHOS_MAX_TASKS'))
print('MYTHOS_TTT_STEPS =', os.environ.get('MYTHOS_TTT_STEPS'))
print('MYTHOS_TTT_NUM_AUG =', os.environ.get('MYTHOS_TTT_NUM_AUG'))
print('MYTHOS_TTT_RANK =', os.environ.get('MYTHOS_TTT_RANK'))
print('MYTHOS_TTT_LR =', os.environ.get('MYTHOS_TTT_LR'))
print('MYTHOS_TTT_BATCH_SIZE =', os.environ.get('MYTHOS_TTT_BATCH_SIZE'))
print('MYTHOS_TTT_GENIE_WEIGHT =', os.environ.get('MYTHOS_TTT_GENIE_WEIGHT'))


## 3. Import Mythos Runtime

In [ ]:
import mythos
from mythos.arc import load_challenges
from mythos.kaggle_run import resolve_challenge_path, resolve_solution_path
from mythos.kaggle_models import (
    autodiscover_model_inputs,
    download_direct_checkpoint_inputs,
    download_git_code_repositories,
    download_huggingface_model_inputs,
)
from mythos.arc import load_solutions
from mythos.metrics import score_files, score_submission_data
from mythos.pipeline import PLAN_STAGE_ORDER
from mythos.solvers.factory import make_solver
from mythos.submission import load_submission, write_submission

from mythos.training import (
    JepaProjectionConfig,
    WorldModelConfig,
    run_ttt_lora_smoke,
    train_jepa_projection,
    train_world_model,
)

print('Imported mythos from:', mythos.__file__)
print('PLAN_STAGE_ORDER =', ' -> '.join(PLAN_STAGE_ORDER))


## 4. Load ARC Data

In [ ]:
challenge_path = resolve_challenge_path(DATA_DIR, SPLIT)
solution_path = resolve_solution_path(DATA_DIR, SPLIT)
tasks = load_challenges(challenge_path)

_max_tasks = os.environ.get('MYTHOS_MAX_TASKS')
if _max_tasks:
    tasks = dict(list(tasks.items())[: int(_max_tasks)])
    print(f'MYTHOS_MAX_TASKS set: truncated to {len(tasks)} task(s) for a fast iteration pass')

train_examples = sum(len(task.train) for task in tasks.values())
test_items = sum(len(task.test) for task in tasks.values())

print('challenge_path =', challenge_path)
print('solution_path =', solution_path)
print('tasks =', len(tasks))
print('train_examples =', train_examples)
print('test_items =', test_items)


## 5. Download Models, Optionally Train Local Stages, Then Load Solver

In [ ]:
if AUTO_DOWNLOAD_GIT_CODE:
    git_download = download_git_code_repositories(apply=True)
    print('git_code_download =')
    print(json.dumps(git_download, indent=2))

if AUTO_DOWNLOAD_HF_MODELS:
    hf_download = download_huggingface_model_inputs(apply=True)
    print('huggingface_model_download =')
    print(json.dumps(hf_download, indent=2))

if AUTO_DOWNLOAD_DIRECT_CHECKPOINTS:
    direct_download = download_direct_checkpoint_inputs(apply=True)
    print('direct_checkpoint_download =')
    print(json.dumps(direct_download, indent=2))

if AUTO_DISCOVER_MODELS:
    discovery = autodiscover_model_inputs(apply=True)
    print('model_autodiscovery =')
    print(json.dumps(discovery, indent=2))

training_results = {}
if RUN_TRAINING_STAGES:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    if TRAIN_JEPA_PROJECTION:
        projection_result = train_jepa_projection(
            tasks.values(),
            checkpoint_path=IJEPA_PROJECTION_OUTPUT,
            config=JepaProjectionConfig(input_dim=1280, output_dim=768),
            steps=JEPA_PROJECTION_STEPS,
            lr=1e-3,
        )
        os.environ['IJEPA_PROJECTION_CHECKPOINT_PATH'] = str(IJEPA_PROJECTION_OUTPUT)
        training_results['jepa_projection'] = projection_result.to_dict()
    if TRAIN_WORLD_MODEL:
        world_result = train_world_model(
            tasks.values(),
            checkpoint_path=WORLD_MODEL_OUTPUT,
            config=WorldModelConfig(z_dim=768, rule_dim=4, hidden_dim=3072),
            steps=WORLD_MODEL_STEPS,
            lr=1e-3,
        )
        os.environ['WORLD_MODEL_CHECKPOINT_PATH'] = str(WORLD_MODEL_OUTPUT)
        training_results['world_model'] = world_result.to_dict()
    if RUN_TTT_SMOKE:
        ttt_result = run_ttt_lora_smoke(
            rank=16,
            steps=TTT_SMOKE_STEPS,
            checkpoint_path=TTT_LORA_OUTPUT,
        )
        os.environ['TTT_LORA_CHECKPOINT_PATH'] = str(TTT_LORA_OUTPUT)
        training_results['ttt_lora_smoke'] = ttt_result.to_dict()
else:
    print('RUN_TRAINING_STAGES is False; using downloaded/discovered checkpoints only.')

if training_results:
    print('training_results =')
    print(json.dumps(training_results, indent=2, sort_keys=True))

solver = make_solver(SOLVER_NAME, model_mode=MODEL_MODE)
print('Loaded solver:', solver.__class__.__name__)

if hasattr(solver, 'pipeline'):
    print('model_registry =')
    print(json.dumps(solver.pipeline.model_registry.summary(), indent=2))


## 5b. Install HRM Runtime Dependencies

The external HRM repo needs packages Kaggle's base image doesn't ship (`flash-attn` in particular has no fallback attention path). This cell installs them best-effort; if anything critical fails, it disables real HRM inference and falls back to the deterministic pipeline solver rather than risk a hung or crashed GPU session.

In [ ]:
hrm_setup_ok = True
hrm_setup_log = []

if SOLVER_NAME == 'hrm':
    import subprocess
    import torch as _torch_probe

    print('torch =', _torch_probe.__version__, '| cuda =', _torch_probe.version.cuda,
          '| cuda_available =', _torch_probe.cuda.is_available(),
          '| device_count =', _torch_probe.cuda.device_count())
    if _torch_probe.cuda.is_available():
        print('gpu_name =', _torch_probe.cuda.get_device_name(0))

    hrm_dependencies = [
        'einops', 'tqdm', 'coolname', 'pydantic', 'argdantic', 'wandb',
        'omegaconf', 'hydra-core', 'huggingface_hub', 'pyyaml',
        # adam-atan2's setup.py uses the legacy setup_requires=['setuptools_scm']
        # auto-fetch path, which pulls a setuptools_scm version whose own
        # vcs_versioning dependency doesn't resolve through that legacy mechanism.
        # Installing setuptools_scm normally first lets adam-atan2's build find it
        # already satisfied and skip the broken auto-fetch.
        'setuptools_scm', 'adam-atan2',
    ]
    # Kaggle's scored rerun is internet-disabled, so PyPI is unreachable too --
    # confirmed by a real run: pydantic/pyyaml/etc already ship in Kaggle's base
    # image (installs no-op fine offline), but coolname/argdantic/hydra-core/
    # setuptools_scm/adam-atan2 don't and failed outright with no internet. Their
    # wheels (pure-python only; platform-specific ones like pydantic-core were
    # deliberately excluded since those packages are already present) are
    # pre-staged in the same dataset as the checkpoint.
    wheelhouse = Path('/kaggle/input/hrm-arc2-checkpoint/wheels')
    print('wheelhouse =', wheelhouse, '| is_dir =', wheelhouse.is_dir())
    hrm_dataset_root = Path('/kaggle/input/hrm-arc2-checkpoint')
    if hrm_dataset_root.is_dir():
        print('hrm-arc2-checkpoint dataset contents:', sorted(p.name for p in hrm_dataset_root.iterdir()))
    else:
        print('hrm-arc2-checkpoint dataset root not found; listing /kaggle/input:')
        print(sorted(p.name for p in Path('/kaggle/input').iterdir()) if Path('/kaggle/input').is_dir() else 'no /kaggle/input')
    offline_install_args = ['--no-index', '--find-links', str(wheelhouse)] if wheelhouse.is_dir() else []

    def _resolve_install_target(package):
        # Resolve to the exact staged .whl instead of relying on pip's --find-links
        # name-based matching: confirmed on a real run that Kaggle's pip 24.1.2 fails
        # to match 'adam-atan2' against a staged sdist by name. The glob is restricted
        # to *.whl files specifically -- Kaggle auto-extracts uploaded .tar.gz archives
        # into a same-named bare directory (confirmed: an old 'adam_atan2-0.0.3/' dir
        # from a superseded dataset version was still present and, unfiltered, sorted
        # ahead of the real wheel and got picked instead), so only .whl is ever staged.
        if not wheelhouse.is_dir():
            return package
        normalized = package.replace('-', '_')
        candidates = sorted(wheelhouse.glob(f'{normalized}-*.whl')) or sorted(wheelhouse.glob(f'{package}-*.whl'))
        return str(candidates[0]) if candidates else package

    failed_packages = []
    for package in hrm_dependencies:
        # Install one at a time: a single bad package must not take down the
        # whole batch and hide which of the others would have installed fine.
        try:
            install = subprocess.run(
                # -v: pip swallows the build subprocess's own traceback by default
                # (even without -q) and only shows its own generic wrapper error;
                # -v is what actually surfaces why a legacy setup.py build failed.
                [sys.executable, '-m', 'pip', 'install', '-v', *offline_install_args, _resolve_install_target(package)],
                capture_output=True, text=True, timeout=300,
            )
            hrm_setup_log.append({'step': f'pip_install:{package}', 'returncode': install.returncode})
            if install.returncode != 0:
                failed_packages.append(package)
                print(f'WARNING: failed to install {package}:')
                print('--- stdout (tail) ---')
                print(install.stdout[-4000:])
                print('--- stderr (tail) ---')
                print(install.stderr[-4000:])
        except subprocess.TimeoutExpired:
            failed_packages.append(package)
            hrm_setup_log.append({'step': f'pip_install:{package}', 'error': 'timed out after 300s'})
            print(f'WARNING: installing {package} timed out after 300s')
    if failed_packages:
        hrm_setup_ok = False
        print('WARNING: HRM dependency install failed for:', failed_packages)

    if hrm_setup_ok:
        # flash-attn has no prebuilt wheel matching Kaggle's exact torch build and a
        # from-source compile was observed taking 100+ minutes without finishing (nvcc
        # compiling many CUDA template instantiations, no fast path available). HRM's
        # Attention.forward() only calls flash_attn_func(q=, k=, v=, causal=) with plain
        # [batch, seq_len, heads, head_dim] tensors -- torch's own built-in
        # scaled_dot_product_attention implements the same math and ships with the stock
        # torch Kaggle already has installed, so no extra install/compile is needed at all.
        import types

        def _flash_attn_func(q, k, v, causal=False, softmax_scale=None, dropout_p=0.0, **_ignored):
            q_, k_, v_ = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
            num_heads, num_kv_heads = q_.shape[1], k_.shape[1]
            if num_kv_heads and num_kv_heads != num_heads:
                repeat = num_heads // num_kv_heads
                k_ = k_.repeat_interleave(repeat, dim=1)
                v_ = v_.repeat_interleave(repeat, dim=1)
            out = _torch_probe.nn.functional.scaled_dot_product_attention(
                q_, k_, v_, dropout_p=dropout_p, is_causal=causal, scale=softmax_scale,
            )
            # .contiguous(): .transpose() is a view (stride swap only); real
            # flash_attn_func returns memory already laid out this way, and the
            # model's own code does a plain .view() right after this call, which
            # requires contiguous memory (confirmed failure: 'Cannot view a tensor
            # with shape ... and strides ...' -- that's the exact non-contiguous
            # symptom -- .reshape() would also work but .contiguous() matches what
            # the real function actually hands back).
            return out.transpose(1, 2).contiguous()

        _flash_attn_shim = types.ModuleType('flash_attn')
        _flash_attn_shim.flash_attn_func = _flash_attn_func
        sys.modules['flash_attn'] = _flash_attn_shim
        hrm_setup_log.append({'step': 'flash_attn_shim', 'note': 'using torch.nn.functional.scaled_dot_product_attention, no compile'})
        print('Installed flash_attn shim backed by scaled_dot_product_attention (no compile needed)')

        # adam_atan2's compiled CUDA/C++ backend (adam_atan2_backend) is only built
        # when torch is present at pip-build time; our offline wheel was built without
        # torch available, so it's missing and 'import pretrain' fails outright (it
        # imports AdamATan2 unconditionally to build the optimizer in init_train_state,
        # even though .step() is never called in this eval-only flow). Provide a
        # faithful pure-PyTorch implementation of the same fused update instead of a
        # whole extra Kaggle round-trip just to compile one C extension.
        def _adam_atan2_cuda_impl_(params, grads, exp_avgs, exp_avg_sqs, state_steps, lr, beta1, beta2, weight_decay):
            for param, grad, exp_avg, exp_avg_sq, step in zip(params, grads, exp_avgs, exp_avg_sqs, state_steps):
                if weight_decay != 0:
                    param.mul_(1 - lr * weight_decay)
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bias_correction1 = 1 - beta1 ** step.item()
                bias_correction2 = 1 - beta2 ** step.item()
                numerator = exp_avg / bias_correction1
                denominator = (exp_avg_sq / bias_correction2).sqrt()
                param.add_(_torch_probe.atan2(numerator, denominator), alpha=-lr)

        _adam_atan2_backend_shim = types.ModuleType('adam_atan2_backend')
        _adam_atan2_backend_shim.adam_atan2_cuda_impl_ = _adam_atan2_cuda_impl_
        sys.modules['adam_atan2_backend'] = _adam_atan2_backend_shim
        hrm_setup_log.append({'step': 'adam_atan2_backend_shim', 'note': 'pure-PyTorch AdamATan2 update, no compile'})
        print('Installed adam_atan2_backend shim (pure PyTorch, no compile needed)')

    if not hrm_setup_ok:
        print('Disabling real HRM inference for this run; falling back to SOLVER_NAME=pipeline.')
        SOLVER_NAME = 'pipeline'
        ENABLE_REAL_HRM_INFERENCE = False
        os.environ['MYTHOS_ENABLE_REAL_HRM'] = '0'
        solver = make_solver(SOLVER_NAME, model_mode=MODEL_MODE)

print('hrm_setup_ok =', hrm_setup_ok)
print('hrm_setup_log =', json.dumps(hrm_setup_log, indent=2))
print('SOLVER_NAME (post-setup) =', SOLVER_NAME)


## 6. Run Plan-Aligned Pipeline

In [ ]:
from mythos.arc import copy_grid
from mythos.solvers.base import make_prediction
from mythos.solvers.baseline import BaselineSolver

def solve_with_fallback(solver, fallback_solver, task):
    # A Kaggle rerun must always produce a submission.json; one task raising
    # must never abort the loop and discard every already-solved prediction.
    try:
        return solver.solve(task)
    except Exception as exc:
        print(f'WARNING: {task.id} failed with {solver.__class__.__name__}: {exc!r}; using baseline fallback')
    try:
        return fallback_solver.solve(task)
    except Exception as exc:
        print(f'WARNING: {task.id} baseline fallback also failed: {exc!r}; using trivial prediction')
    attempts = [(copy_grid(example.input), [[0]]) for example in task.test]
    return make_prediction(task, attempts)

started = time.perf_counter()
predictions = []
fallback_solver = solver if isinstance(solver, BaselineSolver) else BaselineSolver()

if SOLVER_NAME == 'hrm':
    from mythos.solvers.hrm import HRMEnvironment, HRMInferenceRunner, HRMTTTRunner, TTTConfig
    try:
        env = HRMEnvironment.from_env()
        env.validate(require_cuda=True)
        if os.environ.get('MYTHOS_ENABLE_TTT') == '1':
            hrm_runner = HRMTTTRunner(
                env,
                ttt=TTTConfig(
                    rank=int(os.environ.get('MYTHOS_TTT_RANK', '16')),
                    steps=int(os.environ.get('MYTHOS_TTT_STEPS', '20')),
                    lr=float(os.environ.get('MYTHOS_TTT_LR', '1e-3')),
                    batch_size=int(os.environ.get('MYTHOS_TTT_BATCH_SIZE', '2')),
                    genie_weight=float(os.environ.get('MYTHOS_TTT_GENIE_WEIGHT', '0.1')),
                ),
                num_aug=int(os.environ.get('MYTHOS_TTT_NUM_AUG', '0')),
            )
        else:
            hrm_runner = HRMInferenceRunner(env)
        predictions = hrm_runner.solve_tasks(list(tasks.values()))
        print(f'Batched HRM solved {len(predictions)} task(s)')
    except Exception as exc:
        import traceback
        print(f'WARNING: HRM batch run failed: {exc!r}; using baseline fallback for all tasks')
        print('--- full traceback ---')
        traceback.print_exc()
        if hasattr(exc, 'stdout') and exc.stdout:
            print('--- subprocess stdout (tail) ---')
            print(exc.stdout[-4000:])
        if hasattr(exc, 'stderr') and exc.stderr:
            print('--- subprocess stderr (tail) ---')
            print(exc.stderr[-4000:])
        predictions = [fallback_solver.solve(task) for task in tasks.values()]
else:
    for index, task in enumerate(tasks.values(), start=1):
        prediction = solve_with_fallback(solver, fallback_solver, task)
        predictions.append(prediction)
        if index <= 3 or index == len(tasks):
            print(f'{index}/{len(tasks)} solved: {task.id}')

write_submission(predictions, OUTPUT_PATH)
elapsed = time.perf_counter() - started

print('Wrote submission:', OUTPUT_PATH)
print('tasks_predicted =', len(predictions))
print('elapsed_seconds =', round(elapsed, 3))

if hasattr(solver, 'last_trace') and solver.last_trace is not None:
    print('last_pipeline_trace =')
    print(json.dumps(solver.last_trace.to_dict(), indent=2))


## 7. Validate Submission and Score When Solutions Exist

In [ ]:
submission = load_submission(OUTPUT_PATH)
submission_items = sum(len(outputs) for outputs in submission.values())

print('submission_tasks =', len(submission))
print('submission_test_items =', submission_items)
print('sample_task_id =', next(iter(submission)))

if solution_path is not None:
    if _max_tasks:
        # score_files requires every solved task to be present (correct for a
        # real full run); a MYTHOS_MAX_TASKS test run only solved a subset, so
        # score just that subset instead of letting it raise on the rest.
        all_solutions = load_solutions(solution_path)
        partial_solutions = {task_id: all_solutions[task_id] for task_id in submission if task_id in all_solutions}
        score = score_submission_data(submission, partial_solutions)
        print(f'score (partial: {len(partial_solutions)}/{len(all_solutions)} tasks, MYTHOS_MAX_TASKS set) =')
    else:
        score = score_files(str(OUTPUT_PATH), str(solution_path))
        print('score =')
    print(json.dumps(score.to_dict(), indent=2, sort_keys=True))

    # DIAGNOSTIC: print predicted grids next to the true answer for two
    # representative tasks, to see the real failure mode directly instead of
    # guessing from aggregate metrics alone (cell accuracy alone can't say
    # whether it's a wrong output shape, systematically wrong colors, or
    # something else). One task has train output shapes that disagree
    # (output_shape_hint defers to the model's own EOS markers); one has
    # train output shapes that all agree (shape is never in question, so any
    # remaining failure is purely about predicted grid content).
    solutions_for_diagnostic = partial_solutions if _max_tasks else load_solutions(solution_path)
    candidate_ids = [tid for tid in submission if tid in solutions_for_diagnostic]
    def _train_shapes_agree(task_id):
        shapes = {(len(ex.output), len(ex.output[0])) for ex in tasks[task_id].train if ex.output is not None}
        return len(shapes) == 1
    diagnostic_task_ids = []
    variable_shape_id = next((tid for tid in candidate_ids if not _train_shapes_agree(tid)), None)
    fixed_shape_id = next((tid for tid in candidate_ids if _train_shapes_agree(tid)), None)
    if variable_shape_id is not None:
        diagnostic_task_ids.append(('variable-train-shape', variable_shape_id))
    if fixed_shape_id is not None:
        diagnostic_task_ids.append(('fixed-train-shape', fixed_shape_id))
    for label, diagnostic_task_id in diagnostic_task_ids:
        truth = solutions_for_diagnostic[diagnostic_task_id]
        preds_for_task = submission[diagnostic_task_id]
        source_task = tasks[diagnostic_task_id]
        print(f'--- diagnostic ({label}): task {diagnostic_task_id} ---')
        for demo_index, example in enumerate(source_task.train):
            print(f'train[{demo_index}] input =', example.input)
            print(f'train[{demo_index}] output =', example.output)
        for item_index, (prediction, truth_grid) in enumerate(zip(preds_for_task, truth)):
            print(f'test[{item_index}] input =', source_task.test[item_index].input)
            print(f'test[{item_index}] attempt_1 =', prediction.attempt_1)
            print(f'test[{item_index}] attempt_2 =', prediction.attempt_2)
            print(f'test[{item_index}] true output =', truth_grid)
else:
    print('No solutions file for this split; skipping score.')


## 8. Optional HRM Smoke Test

In [ ]:
if RUN_HRM_SMOKE:
    from mythos.hrm_dataset import prepare_hrm_raw_dataset
    from mythos.solvers.hrm import HRMEnvironment

    env = HRMEnvironment.from_env()
    env.validate(require_cuda=True)
    modules = env.import_modules()
    checkpoint = env.load_checkpoint()
    raw_dir = prepare_hrm_raw_dataset(tasks.values(), Path('/kaggle/working/hrm_smoke/raw/ARC-AGI-2/data'))

    print('HRM repo:', env.repo_dir)
    print('HRM checkpoint:', env.checkpoint_path)
    print('Imported modules:', sorted(modules))
    print('Checkpoint type:', type(checkpoint).__name__)
    print('Prepared HRM raw data:', raw_dir)
else:
    print('RUN_HRM_SMOKE is False; skipping HRM smoke test.')
